In [ ]:
# Waveform and Firing Rate Clustering Analysis
# This notebook performs comprehensive clustering analysis combining waveform data with firing rate and bursting metrics

import pynapple as nap
from spikeinterface import load_sorting_analyzer
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import fnmatch
import matplotlib as mpl
import re
import os
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import umap
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully!")

# Bursting capacity metrics based on autocorrelograms
def calculate_autocorrelogram(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate autocorrelogram for spike times.
    
    Parameters:
    - spike_times: array of spike times in seconds
    - bin_size: bin size in seconds (default 1ms)
    - window_size: window size in seconds (default 100ms)
    
    Returns:
    - bins: time bins
    - autocorr: autocorrelogram values
    """
    if len(spike_times) < 2:
        return np.array([0]), np.array([0])
    
    # Create bins
    bins = np.arange(-window_size, window_size + bin_size, bin_size)
    
    # Calculate autocorrelogram
    autocorr = np.zeros(len(bins) - 1)
    
    for i, spike_time in enumerate(spike_times):
        # Find all other spikes
        other_spikes = np.concatenate([spike_times[:i], spike_times[i+1:]])
        
        # Calculate time differences
        time_diffs = other_spikes - spike_time
        
        # Bin the differences
        hist, _ = np.histogram(time_diffs, bins=bins)
        autocorr += hist
    
    # Normalize by number of spikes
    autocorr = autocorr / len(spike_times)
    
    return bins[:-1], autocorr

def calculate_bursting_metrics(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate multiple bursting capacity metrics from autocorrelogram.
    
    Returns a dictionary with various bursting metrics:
    1. Burst Index: ratio of short-interval spikes to long-interval spikes
    2. Refractory Period Violation: spikes in refractory period
    3. Burst Peak Height: height of the first peak after time 0
    4. Burst Peak Width: width of the first peak
    5. Autocorr Skewness: skewness of the autocorrelogram
    6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    """
    if len(spike_times) < 10:  # Need sufficient spikes for reliable metrics
        return {
            'burst_index': 0,
            'refractory_violation': 0,
            'burst_peak_height': 0,
            'burst_peak_width': 0,
            'autocorr_skewness': 0,
            'short_isi_ratio': 0,
            'autocorr_vector': np.zeros(200)  # Fixed size for consistency
        }
    
    # Calculate autocorrelogram
    bins, autocorr = calculate_autocorrelogram(spike_times, bin_size, window_size)
    
    # Find center bin (time = 0)
    center_idx = len(bins) // 2
    
    # 1. Burst Index: ratio of short-interval spikes (1-10ms) to long-interval spikes (50-100ms)
    short_window = (bins >= 0.001) & (bins <= 0.010)  # 1-10ms
    long_window = (bins >= 0.050) & (bins <= 0.100)   # 50-100ms
    
    short_count = np.sum(autocorr[short_window])
    long_count = np.sum(autocorr[long_window])
    burst_index = short_count / (long_count + 1e-10)  # Add small value to avoid division by zero
    
    # 2. Refractory Period Violation: spikes in 0-2ms window
    refractory_window = (bins >= 0) & (bins <= 0.002)
    refractory_violation = np.sum(autocorr[refractory_window])
    
    # 3. Burst Peak Height: height of the first peak after time 0
    # Look for peaks in the 1-20ms window
    peak_window = (bins >= 0.001) & (bins <= 0.020)
    if np.any(peak_window):
        burst_peak_height = np.max(autocorr[peak_window])
    else:
        burst_peak_height = 0
    
    # 4. Burst Peak Width: width at half height of the first peak
    if burst_peak_height > 0:
        half_height = burst_peak_height / 2
        # Fix the indexing issue - use the correct window for peak_indices
        peak_indices = np.where(autocorr[peak_window] >= half_height)[0]
        if len(peak_indices) > 0:
            burst_peak_width = (np.max(peak_indices) - np.min(peak_indices)) * bin_size
        else:
            burst_peak_width = 0
    else:
        burst_peak_width = 0
    
    # 5. Autocorr Skewness: skewness of the autocorrelogram
    autocorr_skewness = stats.skew(autocorr)
    
    # 6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    isis = np.diff(spike_times)
    short_isis = np.sum(isis < 0.010)
    total_isis = len(isis)
    short_isi_ratio = short_isis / (total_isis + 1e-10)
    
    # Create a standardized autocorr vector (fixed size for consistency)
    autocorr_vector = np.zeros(200)
    if len(autocorr) > 0:
        # Interpolate to fixed size
        autocorr_vector = np.interp(np.linspace(0, len(autocorr)-1, 200), 
                                  np.arange(len(autocorr)), autocorr)
    
    return {
        'burst_index': burst_index,
        'refractory_violation': refractory_violation,
        'burst_peak_height': burst_peak_height,
        'burst_peak_width': burst_peak_width,
        'autocorr_skewness': autocorr_skewness,
        'short_isi_ratio': short_isi_ratio,
        'autocorr_vector': autocorr_vector
    }

print("Bursting metrics functions defined")

def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')

In [ ]:
#subject definitions for tumor cohort
nwb_paths = [
    Path("/data_store2/neuropixels/nwb/old/NP93_B1/NP93_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP95_B1/NP95_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP101_B3/NP101_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP105_B1/NP105_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP113_B1/NP113_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP114_B1/NP114_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP116_B2/NP116_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP122_B1/NP122_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP128_B1/NP128_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP129_B1/NP129_B1.nwb"),
    #Path("/data_store2/neuropixels/nwb/old/NP132_B2/NP132_B2.nwb"),
    #Path("/data_store2/neuropixels/nwb/old/NP132_B3/NP132_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP136_B1/NP136_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP137_B1/NP137_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP138_B1/NP138_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B1/NP139_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B2/NP139_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP147_B2/NP147_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP150_B1/NP150_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP153_B1/NP153_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP171_B1/NP171_B1.nwb"), # new gbm flair-
    Path("/data_store2/neuropixels/nwb/old/NP174_B3/NP174_B3.nwb"), # new gbm flair-
]   

### adjust for the depth of the probe
subj_list = [1, 2, 3, 5, 6, 6, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15, 16, 16, 17, 17, 18, 19, 20, 20, 21, 22, 23]
path_list = ['ast', 'ast', 'gbm','oli', 'gbm', 'gbm', 'gbm', 'gbm', 'gbm', 'ast', 'ast', 'ast', 'ast', 'oli', 'oli', 'gbm', 'ast', 'ast', 'ast', 'ast', 'ast', 'ast', 'gbm', 'ast', 'ast', 'oli', 'gbm', 'gbm']
grade_list = [4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 2, 2, 4, 2, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 2, 3, 4, 4]
yield_list = [1, 57, 17, 1, 139, 178, 202, 49, 4, 30, 9, 29, 3, 77, 126, 30, 35, 10, 1, 34, 8, 42, 30, 10, 11, 22, 17, 24]
opercular_list = [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
region_list = ['aSTG','SFG','aSTG','SFG','vPrCG','vPrCG','vPrCG','pSTG','aMTG','MFG','aSTG','parsOp','MFG','PoCG','PoCG','pSTG','parsOr','vPrCG','pSTG','parsTr','pSTG','parsTr','SMG','parsTr','pSTG','vPrCG', 'aSTG', 'vPrCG']
age_list = [28, 28, 54, 34, 52, 52, 52, 59, 64, 34, 34, 34, 39, 43, 43, 63, 38, 43, 43, 29, 29, 29, 47, 31, 31, 41, 79, 55]
gender_list = [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0]

manual_exclude_lists = [
    [2], # NP93_B1.imec0
    [], # NP95_B1.imec0
    [29, 45, 55], # NP101_B3.imec0
    [130, 150, 151, 245, 371, 375, 377, 380, 386, 452], # NP105_B1.imec0
    [149, 156, 172, 173, 176, 217, 221, 253, 272, 276, 336, 487], # NP113_B1.imec0
    [22, 112, 149, 155, 161, 167, 176, 188, 190, 196, 213, 216, 241, 252, 255, 258, 286, 289, 307, 325, 326, 370, 395, 442, 443, 448, 449, 451, 455, 510, 549, 582, 692, 710, 717, 719, 722, 726, 729], # NP113_B1.imec1
    [385, 448], # NP113_B2.imec2
    [32, 34, 37, 39, 67, 142, 145, 175, 188, 310, 325, 331, 391, 392], # NP114_B1.imec0
    [11, 15, 30, 36], # NP116_B2.imec0
    [340, 382], # NP122_B1.imec0
    [54, 109, 152], # NP128_B1.imec0
    [], # NP128_B1.imec1
    [0, 6, 13, 33, 51, 52], # NP129_B1.imec0
    [19, 62, 111, 151, 171, 192, 199, 200, 205, 210, 227, 229, 244, 290, 298, 311], # NP132_B2.imec0
    [0, 10, 11, 15, 29, 45, 61, 112, 127, 149, 151, 158, 159, 172, 197, 209, 212, 218, 297, 315], # NP132_B3.imec0
    [334, 335, 333], # NP136_B1.imec0
    [361], # NP137_B1.imec0
    [], # NP138_B1.imec0
    [169, 195, 198], # NP138_B1.imec1
    [137, 149, 196, 197, 199, 206, 212, 220, 223, 231], # NP139_B1.imec0
    [129, 248, 249], # NP139_B1.imec1
    [28, 59, 60, 62, 63, 82, 104, 112, 146], # NP139_B2.imec0
    [47, 57, 59, 73, 82, 88, 116, 124, 125, 126, 127, 129, 148, 175, 194, 196, 239, 240, 242, 243, 244, 246, 253, 267, 245, 247, 248, 249, 250, 251, 254], # NP147_B2.imec0
    [], # NP150_B1.imec0
    [34, 64, 119, 135], # NP150_B1.imec1
    [20, 22, 47, 71, 80, 118, 125, 134, 141, 145, 146, 166, 173, 176, 191, 270, 298, 307, 385, 414, 391, 472, 473, 490, 491], # NP153_B1.imec0
    [16, 67, 74, 78, 91, 99, 221, 224, 225, 288], # NP171_B1.imec0
    [9, 11, 13, 23, 40, 43, 47, 59, 67, 72, 87, 104, 109, 122, 166, 169, 172, 175, 197, 199, 203, 205, 211, 212, 219, 227, 229, 230, 235, 237, 240, 251, 256, 295, 296], # NP174_B3.imec0
]

tipDepth_list = [
    5000, # NP93_B1.imec0
    6600, # NP95_B1.imec0
    5843, # NP101_B3.imec0
    7200, # NP105_B1.imec0
    7600, # NP113_B1.imec0
    7600, # NP113_B1.imec1
    7600, # NP113_B1.imec2
    6900, # NP114_B1.imec0
    6100, # NP116_B2.imec0
    6880, # NP122_B1.imec0
    6200, # NP128_B1.imec0
    6200, # NP128_B1.imec1
    6600, # NP129_B1.imec0
    3820, # NP132_B2.imec0
    3820, # NP132_B3.imec0
    6620, # NP136_B1.imec0
    7000, # NP137_B1.imec0
    7100, # NP138_B1.imec0
    7400, # NP138_B1.imec1
    6900, # NP139_B1.imec0
    7400, # NP139_B1.imec1
    7000, # NP139_B2.imec0
    6400, # NP147_B2.imec0
    6840, # NP150_B1.imec0
    6500, # NP150_B1.imec1
    6500, # NP153_B1.imec0
    6800, # NP171_B1.imec0
    5640, # NP174_B3.imec0
]

# Exclude PoCG entries since not using since don't have depth info on them since used dense montage
exclude_indices = [13, 14] #subject definitions for tumor cohort

def exclude_indices_from_list(the_list, indices):
    return [item for idx, item in enumerate(the_list) if idx not in indices]

subj_list = exclude_indices_from_list(subj_list, exclude_indices)
path_list = exclude_indices_from_list(path_list, exclude_indices)
grade_list = exclude_indices_from_list(grade_list, exclude_indices)
yield_list = exclude_indices_from_list(yield_list, exclude_indices)
opercular_list = exclude_indices_from_list(opercular_list, exclude_indices)
region_list = exclude_indices_from_list(region_list, exclude_indices)
age_list = exclude_indices_from_list(age_list, exclude_indices)
gender_list = exclude_indices_from_list(gender_list, exclude_indices)
manual_exclude_lists = exclude_indices_from_list(manual_exclude_lists, exclude_indices)
tipDepth_list = exclude_indices_from_list(tipDepth_list, exclude_indices)

print(f"Loaded {len(nwb_paths)} NWB files")
print(len(subj_list))
print(len(grade_list))
print(len(path_list))
print(len(yield_list))
print(len(opercular_list))
print(len(region_list))
print(len(age_list))
print(len(gender_list))
print(len(manual_exclude_lists))
print(len(tipDepth_list))
print(region_list)
print(subj_list)

In [ ]:
# MAIN LOOP TO EXTRACT WAVEFORMS, FIRING RATES, AND COMPREHENSIVE METRICS
# If summed behavioral duration > 10 min: restrict to beh epochs (unchanged logic).
# If < 10 min: use full recording (no restrict); KS uses recording time span.

print("Starting data extraction from all sessions...")

MIN_BEHAVIOR_MINUTES = 0

def _behavior_duration_seconds(task_times_obj):
    starts = np.asarray(task_times_obj.start, dtype=float).ravel()
    ends = np.asarray(task_times_obj.end, dtype=float).ravel()
    if starts.size != ends.size:
        return np.nan
    return float(np.sum(ends - starts))

def _ks_time_bounds_full_recording(spike_times_obj):
    """Min/max time for uniform KS (full recording)."""
    # span of all units' spike times
    tmin, tmax = np.inf, -np.inf
    for u in range(len(spike_times_obj)):
        idx = spike_times_obj[u].as_series().index.values
        if len(idx):
            tmin = min(tmin, float(np.min(idx)))
            tmax = max(tmax, float(np.max(idx)))
    if np.isfinite(tmin) and np.isfinite(tmax):
        return tmin, tmax
    return np.nan, np.nan

# Initialize data storage lists
indicesAll = []
firingRatesAll = []
waveformAll = []
depthAll = []
burstingMetricsAll = []
autocorrVectorsAll = []
spikeTimesAll = []

insertion = 0

for i in range(len(nwb_paths)):
    print(f"\nProcessing session {i+1}/{len(nwb_paths)}: {nwb_paths[i].name}")

    try:
        data = nap.load_file(nwb_paths[i])
        keys = data.keys()

        template = "*imec*"
        keys = [key for key in keys if fnmatch.fnmatch(key, template)]
        ks_keys = [key for key in keys if "KS4" in key]
        if ks_keys:
            keys = ks_keys
        th8_keys = [key for key in keys if "Th=8" in key]
        if th8_keys:
            keys = th8_keys
        else:
            th_keys = [key for key in keys if "Th=" in key]
            if th_keys:
                keys = th_keys
        template_sentgen = "*sentgen*"
        template_auto = "*_auto*"
        keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
        if ("NP137" in str(nwb_paths[i])) or ("NP139_B2" in str(nwb_paths[i])):
            keys = [key for key in keys if "imec1" not in key]
        keys = sorted(keys, key=imec_key_sorter)

        for s in range(len(keys)):
            print(f"Processing {keys[s]}")

            spike_times = data[keys[s]]
            firingRates_all = spike_times.metadata["rate"]

            if "TaskTimes" in data.keys():
                task_times = data["TaskTimes"]
            else:
                task_times = data["task_times"]

            start_time = task_times.start
            end_time = task_times.end
            beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)

            beh_sec = _behavior_duration_seconds(task_times)
            beh_min = beh_sec / 60.0 if np.isfinite(beh_sec) else np.nan
            use_full_recording = (not np.isfinite(beh_min)) or (beh_min < MIN_BEHAVIOR_MINUTES)

            if use_full_recording:
                spike_times_beh = spike_times
                firingRates_beh = firingRates_all
                min_time, max_time = _ks_time_bounds_full_recording(spike_times)
                if not (np.isfinite(min_time) and np.isfinite(max_time)) or (max_time <= min_time):
                    print(f"  Short behavior ({beh_min:.2f} min) but could not get recording bounds; skipping shank")
                    insertion += 1
                    continue
                print(
                    f"  Behavioral duration {beh_min:.2f} min (< {MIN_BEHAVIOR_MINUTES} min) — using full recording for filters; KS window [{min_time:.3f}, {max_time:.3f}] s"
                )
            else:
                spike_times_beh = spike_times.restrict(beh_epochs)
                firingRates_beh = spike_times_beh.metadata["rate"]
                min_time = start_time[0]
                max_time = end_time[-1]
                print(
                    f"  Behavioral duration {beh_min:.2f} min (>= {MIN_BEHAVIOR_MINUTES} min) — using TaskTimes restrict; KS window [{min_time:.3f}, {max_time:.3f}] s"
                )

            # Quality checks (KS test, violation percentage)
            ks_stats = np.zeros(len(spike_times))
            ks_pvals = np.zeros(len(spike_times))
            for u in range(len(spike_times)):
                test = spike_times[u].as_series().index.values
                if len(test) > 1:
                    normalized_spike_times = (test - min_time) / (max_time - min_time)
                    ks_result = kstest(normalized_spike_times, "uniform")
                    ks_stats[u] = ks_result.statistic
                    ks_pvals[u] = ks_result.pvalue
                else:
                    ks_stats[u] = np.nan
                    ks_pvals[u] = np.nan

            violationThreshold = 3 / 1000
            violationPct = np.zeros(len(spike_times))
            for u in range(len(spike_times)):
                unit = spike_times[u]
                unit = unit.as_series().index
                if len(unit) < 100:
                    violationPct[u] = 1
                else:
                    isi = unit.diff()[1:]
                    violations = np.where(isi < violationThreshold)
                    violations = np.array(violations)
                    violationPct[u] = violations.size / len(isi)

            if "KSLabel" in spike_times.metadata:
                KSLabels = spike_times.metadata["KSLabel"]
            else:
                KSLabels = spike_times.metadata["quality"]

            firingRates = firingRates_beh
            mask1 = violationPct < 3 / 100
            mask2 = firingRates > 0.5
            mask3 = KSLabels != "noise"
            mask4 = ks_stats < 0.3
            mask = mask1 & mask2 & mask3 & mask4
            indicesFinal = firingRates.index[mask]

            indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion])
            insertion += 1
            spike_times_good = spike_times[indicesFinal]
            print(f"Number of good neurons: {len(spike_times_good)}")

            if len(spike_times_good) == 0:
                print("No good neurons found, skipping session")
                continue

            kilosort_run = f"{keys[s]}_sparse"
            rec_id = os.path.basename(nwb_paths[i])
            rec_id = os.path.splitext(rec_id)[0]
            # If kilosort_run starts with 'catgt_', remove the prefix
            if kilosort_run.startswith("catgt_"):
                kilosort_run = kilosort_run[len("catgt_"):]
   
            analyzer_path = f"/data_store2/neuropixels/nwb/old/{rec_id}/SI/{kilosort_run}"
            
            if not os.path.exists(analyzer_path):
                print(f"Analyzer path not found: {analyzer_path}")
                continue

            analyzer = load_sorting_analyzer(analyzer_path)
            templates = analyzer.get_extension("templates")
            avg_templates = templates.get_data(operator="average")

            waveform_list = []
            depth_list = []
            firing_rates_list = []
            bursting_metrics_list = []
            autocorr_vectors_list = []
            spike_times_list = []

            for unit_id in indicesFinal:
                template = avg_templates[unit_id]
                max_amp_per_channel = np.max(np.abs(template), axis=0)
                max_amp_ch = np.argmax(max_amp_per_channel)
                waveform = template[:, max_amp_ch]
                waveform_list.append(waveform)

                try:
                    depths = analyzer.get_extension("unit_locations").get_data()[:, 1]
                    depth = tipDepth_list[insertion - 1] - depths[unit_id]
                except Exception:
                    depth = -999

                depth_list.append(depth)

                firing_rate = firingRates.loc[unit_id]
                firing_rates_list.append(firing_rate)

                spike_times_unit = spike_times_good[unit_id].as_series().index.values
                bursting_metrics = calculate_bursting_metrics(spike_times_unit)
                bursting_metrics_list.append(bursting_metrics)
                autocorr_vectors_list.append(bursting_metrics["autocorr_vector"])

                spike_times_list.append(spike_times_unit)

            indicesAll.append(indicesFinal)
            firingRatesAll.append(firing_rates_list)
            waveformAll.append(waveform_list)
            depthAll.append(depth_list)
            burstingMetricsAll.append(bursting_metrics_list)
            autocorrVectorsAll.append(autocorr_vectors_list)
            spikeTimesAll.append(spike_times_list)

    except Exception as e:
        print(f"Error processing session {i+1}: {e}")
        import traceback

        traceback.print_exc()
        continue

print(f"\nData extraction complete!")
print(f"Processed {insertion} sessions successfully")
print(f"Total waveforms: {sum(len(w) for w in waveformAll)}")
print(f"Total firing rate measurements: {sum(len(f) for f in firingRatesAll)}")
print(f"Total bursting metric measurements: {sum(len(b) for b in burstingMetricsAll)}")
print(f"Total spike time measurements: {sum(len(s) for s in spikeTimesAll)}")

In [ ]:
# SAVE extracted waveform / metrics (same structure as LOAD SAVED EXTRACTED DATA cell)
# Run once after the main extraction loop completes.

import pickle
from pathlib import Path

save_path = Path("revision_waveform_tumor_v3.pkl")

to_save = {
    "indicesAll": indicesAll,
    "firingRatesAll": firingRatesAll,
    "waveformAll": waveformAll,
    "depthAll": depthAll,
    "burstingMetricsAll": burstingMetricsAll,
    "autocorrVectorsAll": autocorrVectorsAll,
    "spikeTimesAll": spikeTimesAll,
    "insertion": insertion,
}

with open(save_path, "wb") as f:
    pickle.dump(to_save, f, protocol=pickle.HIGHEST_PROTOCOL)

mb = save_path.stat().st_size / (1024 ** 2)
print(f"Saved {save_path.resolve()} ({mb:.2f} MB)")
print(f"  insertion (sessions in lists): {insertion}")
print(f"  len(indicesAll): {len(indicesAll)}")

In [ ]:
# LOAD SAVED EXTRACTED DATA
# Run this cell instead of re-running the data extraction cell above
# This loads the previously extracted data from extracted_data.pkl

import pickle
from pathlib import Path

load_path = Path('revision_waveform_tumor_v3.pkl')

if load_path.exists():
    print(f"Loading saved data from {load_path}...")
    with open(load_path, 'rb') as f:
        saved_data = pickle.load(f)
    
    # Restore all variables
    indicesAll = saved_data['indicesAll']
    firingRatesAll = saved_data['firingRatesAll']
    waveformAll = saved_data['waveformAll']
    depthAll = saved_data['depthAll']
    burstingMetricsAll = saved_data['burstingMetricsAll']
    autocorrVectorsAll = saved_data['autocorrVectorsAll']
    spikeTimesAll = saved_data['spikeTimesAll']
    insertion = saved_data.get('insertion', len(indicesAll))  # Use saved value or fallback
    
    print(f"✓ Data loaded successfully!")
    print(f"  Processed {insertion} sessions")
    print(f"  Total waveforms: {sum(len(w) for w in waveformAll)}")
    print(f"  Total firing rate measurements: {sum(len(f) for f in firingRatesAll)}")
    print(f"  Total bursting metric measurements: {sum(len(b) for b in burstingMetricsAll)}")
    print(f"  Total spike time measurements: {sum(len(s) for s in spikeTimesAll)}")
    print(f"  File size: {load_path.stat().st_size / (1024**2):.2f} MB")
else:
    print(f"⚠️  Warning: {load_path} not found!")
    print("  Please run the data extraction cell above first to create the saved data file.")
    print("  Or check that the file path is correct.")


In [ ]:
# histogram plot
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

plt.figure(figsize=(8,5))
plt.hist([d for session_depths in depthAll for d in session_depths], bins=50, color='skyblue', edgecolor='k')
plt.xlabel('Depth')
plt.ylabel('Count')
plt.title('Histogram of All Depths')
#plt.savefig("depth_histogram.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Find the insertion indices where any neuron has depth > 0
insertion_ids_with_depth_gt_0 = []
for i, session_depths in enumerate(depthAll):
    if any(d < 0 for d in session_depths):
        insertion_ids_with_depth_gt_0.append(i)
print("Insertion IDs with depth > 0:", insertion_ids_with_depth_gt_0)

# Find the indices (in terms of insertion index and neuron index) where neuron depth > 0
# Also show region and opercular status
neuron_ids_with_depth_gt_0 = []
for insertion_idx, session_depths in enumerate(depthAll):
    for neuron_idx, d in enumerate(session_depths):
        if d < 0:
            # Try to extract the original unit_id from indicesAll if available
            try:
                unit_id = indicesAll[insertion_idx][neuron_idx]
            except Exception:
                unit_id = None
            # Try to extract region and opercular status
            try:
                region = region_list[insertion_idx]
            except Exception:
                region = None
            try:
                opercular_status = opercular_list[insertion_idx]
            except Exception:
                opercular_status = None

            neuron_ids_with_depth_gt_0.append({
                'insertion_idx': insertion_idx,
                'neuron_idx': neuron_idx,
                'unit_id_in_spike_times': unit_id,
                'region': region,
                'opercular': opercular_status
            })
print("Neuron (insertion_idx, neuron_idx, unit_id_in_spike_times, region, opercular) with depth > 0:")
for n in neuron_ids_with_depth_gt_0:
    print(n)


In [ ]:
# CLUSTERING ANALYSIS - prepare data > define metrics > collect and standardize > cluster



# 1. prepare data
print("\n=== PREPARING DATA FOR CLUSTERING ===")

# Flatten all data into single arrays
waveforms_flat = []
firing_rates_flat = []
bursting_metrics_flat = []
autocorr_vectors_flat = []
spike_times_flat = []  # NEW: Flatten spike times
metadata_flat = []

for insertion_idx in range(len(waveformAll)):
    for neuron_idx in range(len(waveformAll[insertion_idx])):
        waveforms_flat.append(waveformAll[insertion_idx][neuron_idx])
        firing_rates_flat.append(firingRatesAll[insertion_idx][neuron_idx])
        bursting_metrics_flat.append(burstingMetricsAll[insertion_idx][neuron_idx])
        autocorr_vectors_flat.append(autocorrVectorsAll[insertion_idx][neuron_idx])
        spike_times_flat.append(spikeTimesAll[insertion_idx][neuron_idx])  # NEW
        
        metadata_flat.append({
            'insertion_idx': insertion_idx,
            'unit_id': indicesAll[insertion_idx][neuron_idx],  # Store unit_id for proper matching
            'subj': subj_list[insertion_idx],
            'grade': grade_list[insertion_idx],
            'pathology': path_list[insertion_idx],
            'yield': yield_list[insertion_idx],
            'opercular': opercular_list[insertion_idx],
            'region': region_list[insertion_idx],
            'age': age_list[insertion_idx],
            'gender': gender_list[insertion_idx],
            'depth': depthAll[insertion_idx][neuron_idx]
        })

# Convert to numpy arrays
waveforms_array = np.array(waveforms_flat)
firing_rates_array = np.array(firing_rates_flat)
autocorr_vectors_array = np.array(autocorr_vectors_flat)

print(f"Total neurons: {len(waveforms_flat)}")
print(f"Waveform array shape: {waveforms_array.shape}")
print(f"Firing rates array shape: {firing_rates_array.shape}")
print(f"Autocorr vectors array shape: {autocorr_vectors_array.shape}")
print(f"Spike times data: {len(spike_times_flat)} neurons")  # NEW

# Extract individual bursting metrics
burst_indices = np.array([bm['burst_index'] for bm in bursting_metrics_flat])
refractory_violations = np.array([bm['refractory_violation'] for bm in bursting_metrics_flat])
burst_peak_heights = np.array([bm['burst_peak_height'] for bm in bursting_metrics_flat])
burst_peak_widths = np.array([bm['burst_peak_width'] for bm in bursting_metrics_flat])
autocorr_skewness = np.array([bm['autocorr_skewness'] for bm in bursting_metrics_flat])
short_isi_ratios = np.array([bm['short_isi_ratio'] for bm in bursting_metrics_flat])

print(f"\nBursting metrics extracted:")
print(f"Burst indices: mean={np.mean(burst_indices):.3f}, std={np.std(burst_indices):.3f}")
print(f"Refractory violations: mean={np.mean(refractory_violations):.3f}, std={np.std(refractory_violations):.3f}")
print(f"Burst peak heights: mean={np.mean(burst_peak_heights):.3f}, std={np.std(burst_peak_heights):.3f}")
print(f"Burst peak widths: mean={np.mean(burst_peak_widths):.3f}, std={np.std(burst_peak_widths):.3f}")
print(f"Autocorr skewness: mean={np.mean(autocorr_skewness):.3f}, std={np.std(autocorr_skewness):.3f}")
print(f"Short ISI ratios: mean={np.mean(short_isi_ratios):.3f}, std={np.std(short_isi_ratios):.3f}")

# Standardize waveforms
waveform_scaler = StandardScaler()
waveforms_scaled = waveform_scaler.fit_transform(waveforms_array)
print(f"\nWaveforms standardized: shape={waveforms_scaled.shape}")

# Prepare spiking data (firing rates + bursting metrics)
spiking_data = np.column_stack([
    firing_rates_array,
    burst_indices,
    refractory_violations,
    burst_peak_heights,
    burst_peak_widths,
    autocorr_skewness,
    short_isi_ratios
])

# Standardize spiking data
spiking_scaler = StandardScaler()
spiking_data_scaled = spiking_scaler.fit_transform(spiking_data)
print(f"Spiking data standardized: shape={spiking_data_scaled.shape}")

# Prepare combined data (waveforms + spiking metrics)
# First standardize each component separately, then combine
combined_data = np.column_stack([
    waveforms_scaled,
    spiking_data_scaled
])

print(f"Combined data prepared: shape={combined_data.shape}")
print("Data preparation complete!")



### 2. Define metrics
# COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS
# Based on literature review for distinguishing interneurons vs pyramidal neurons

print("\n=== COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS ===")
print("Selected metrics based on literature review:")
print("Waveform: Spike Width, Amplitude, Asymmetry, Rise Time, Decay Time")
print("Firing Rate: Mean Firing Rate, Burst Index, ISI CV, ISI Violation Rate, Spike Frequency Adaptation")

# Import additional libraries for comprehensive analysis
from sklearn.mixture import GaussianMixture
from scipy.stats import chi2_contingency, fisher_exact
import matplotlib.patches as mpatches

def calculate_spike_width(waveform):
    """Calculate spike width (trough-to-peak duration)"""
    # Find peak and trough
    peak_idx = np.argmax(waveform)
    trough_idx = np.argmin(waveform)
    
    # Calculate width in samples (assuming 30kHz sampling rate)
    width_samples = abs(peak_idx - trough_idx)
    width_ms = width_samples / 30.0  # Convert to milliseconds
    
    return width_ms

def calculate_spike_amplitude(waveform):
    """Calculate spike amplitude (peak-to-trough amplitude).
    Returns positive if abs(peak) > abs(trough), negative if abs(trough) > abs(peak)."""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    amplitude = peak_amp - trough_amp
    if abs(trough_amp) > abs(peak_amp):
        amplitude = -abs(amplitude)
    return amplitude

def calculate_spike_asymmetry(waveform):
    """Calculate spike asymmetry (peak/trough ratio)"""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    
    # Avoid division by zero
    if abs(trough_amp) < 1e-10:
        return np.inf
    
    asymmetry = abs(peak_amp / trough_amp)
    return asymmetry

def calculate_spike_rise_time(waveform):
    """Calculate spike rise time (baseline to peak)"""
    # Find baseline (first 10% of waveform)
    baseline_end = len(waveform) // 10
    baseline = np.mean(waveform[:baseline_end])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going up
    rising_phase = waveform[:peak_idx]
    baseline_crossings = np.where(np.diff(np.sign(rising_phase - baseline)) > 0)[0]
    
    if len(baseline_crossings) > 0:
        rise_start = baseline_crossings[-1]  # Last crossing before peak
        rise_time_samples = peak_idx - rise_start
        rise_time_ms = rise_time_samples / 30.0  # Convert to milliseconds
    else:
        rise_time_ms = peak_idx / 30.0  # Fallback
    
    return rise_time_ms

def calculate_spike_decay_time(waveform):
    """Calculate spike decay time (peak to baseline)"""
    # Find baseline (last 10% of waveform)
    baseline_start = int(len(waveform) * 0.9)
    baseline = np.mean(waveform[baseline_start:])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going down after peak
    decay_phase = waveform[peak_idx:]
    baseline_crossings = np.where(np.diff(np.sign(decay_phase - baseline)) < 0)[0]
    
    if len(baseline_crossings) > 0:
        decay_end = baseline_crossings[0]  # First crossing after peak
        decay_time_samples = decay_end
        decay_time_ms = decay_time_samples / 30.0  # Convert to milliseconds
    else:
        decay_time_ms = (len(waveform) - peak_idx) / 30.0  # Fallback
    
    return decay_time_ms

def calculate_isi_cv(spike_times):
    """Calculate ISI coefficient of variation"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    if len(isis) == 0:
        return 0
    
    mean_isi = np.mean(isis)
    if mean_isi == 0:
        return 0
    
    cv = np.std(isis) / mean_isi
    return cv

def calculate_isi_violation_rate(spike_times, refractory_period=0.002):
    """Calculate ISI violation rate (spikes within refractory period)"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    violations = np.sum(isis < refractory_period)
    total_spikes = len(spike_times)
    
    violation_rate = violations / total_spikes if total_spikes > 0 else 0
    return violation_rate

def calculate_spike_frequency_adaptation(spike_times, window_size=1.0):
    """Calculate spike frequency adaptation"""
    if len(spike_times) < 10:
        return 0
    
    # Divide spike train into windows
    total_time = spike_times[-1] - spike_times[0]
    n_windows = int(total_time / window_size)
    
    if n_windows < 2:
        return 0
    
    window_firing_rates = []
    for i in range(n_windows):
        window_start = spike_times[0] + i * window_size
        window_end = window_start + window_size
        
        spikes_in_window = np.sum((spike_times >= window_start) & (spike_times < window_end))
        firing_rate = spikes_in_window / window_size
        window_firing_rates.append(firing_rate)
    
    if len(window_firing_rates) < 2:
        return 0
    
    # Calculate adaptation as decrease in firing rate over time
    early_rate = np.mean(window_firing_rates[:len(window_firing_rates)//2])
    late_rate = np.mean(window_firing_rates[len(window_firing_rates)//2:])
    
    if early_rate == 0:
        return 0
    
    adaptation = (early_rate - late_rate) / early_rate
    return adaptation

print("Spike characteristic calculation functions defined!")



### 3. Collect and standarize spike metrics
# COLLECT COMPREHENSIVE SPIKE CHARACTERISTICS DATA (CORRECTED VERSION)
print("\n=== COLLECTING COMPREHENSIVE SPIKE CHARACTERISTICS ===")

# Initialize lists for comprehensive characteristics
comprehensive_waveform_metrics = []
comprehensive_firing_rate_metrics = []
comprehensive_metadata = []

print("Processing all neurons for comprehensive spike characteristics...")

# Process each neuron to extract comprehensive metrics
for neuron_idx in range(len(waveforms_flat)):
    if neuron_idx % 100 == 0:
        print(f"Processing neuron {neuron_idx+1}/{len(waveforms_flat)}")
    
    # Get waveform
    waveform = waveforms_flat[neuron_idx]
    
    # Calculate waveform characteristics
    spike_width = calculate_spike_width(waveform)
    spike_amplitude = calculate_spike_amplitude(waveform)
    spike_asymmetry = calculate_spike_asymmetry(waveform)
    spike_rise_time = calculate_spike_rise_time(waveform)
    spike_decay_time = calculate_spike_decay_time(waveform)
    
    waveform_metrics = [spike_width, spike_amplitude, spike_asymmetry, spike_rise_time, spike_decay_time]
    comprehensive_waveform_metrics.append(waveform_metrics)
    
    # Get firing rate characteristics
    firing_rate = firing_rates_flat[neuron_idx]
    burst_index = bursting_metrics_flat[neuron_idx]['burst_index']
    
    # Calculate additional firing rate characteristics using actual spike times
    spike_times_unit = spike_times_flat[neuron_idx]
    
    # Calculate ISI CV from actual spike times
    isi_cv = calculate_isi_cv(spike_times_unit)
    
    # Calculate ISI violation rate from actual spike times
    isi_violation_rate = calculate_isi_violation_rate(spike_times_unit)
    
    # Calculate spike frequency adaptation from actual spike times
    spike_frequency_adaptation = calculate_spike_frequency_adaptation(spike_times_unit)
    
    firing_rate_metrics = [firing_rate, burst_index, isi_cv, isi_violation_rate, spike_frequency_adaptation]
    comprehensive_firing_rate_metrics.append(firing_rate_metrics)
    
    # Store metadata
    comprehensive_metadata.append(metadata_flat[neuron_idx])

# Convert to numpy arrays
comprehensive_waveform_array = np.array(comprehensive_waveform_metrics)
comprehensive_firing_rate_array = np.array(comprehensive_firing_rate_metrics)

print(f"\nComprehensive metrics collected:")
print(f"Waveform metrics shape: {comprehensive_waveform_array.shape}")
print(f"Firing rate metrics shape: {comprehensive_firing_rate_array.shape}")

# Display summary statistics
waveform_metric_names = ['Spike Width (ms)', 'Spike Amplitude', 'Spike Asymmetry', 'Rise Time (ms)', 'Decay Time (ms)']
firing_rate_metric_names = ['Firing Rate (Hz)', 'Burst Index', 'ISI CV', 'ISI Violation Rate', 'Spike Frequency Adaptation']

print("\nWaveform Metrics Summary:")
for i, name in enumerate(waveform_metric_names):
    mean_val = np.mean(comprehensive_waveform_array[:, i])
    std_val = np.std(comprehensive_waveform_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nFiring Rate Metrics Summary:")
for i, name in enumerate(firing_rate_metric_names):
    mean_val = np.mean(comprehensive_firing_rate_array[:, i])
    std_val = np.std(comprehensive_firing_rate_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nComprehensive data collection complete!")

# STANDARDIZE COMPREHENSIVE METRICS AND PREPARE FOR CLUSTERING
print("\n=== STANDARDIZING COMPREHENSIVE METRICS ===")

# Standardize waveform metrics
waveform_scaler_comprehensive = StandardScaler()
comprehensive_waveform_scaled = waveform_scaler_comprehensive.fit_transform(comprehensive_waveform_array)

# Standardize firing rate metrics
firing_rate_scaler_comprehensive = StandardScaler()
comprehensive_firing_rate_scaled = firing_rate_scaler_comprehensive.fit_transform(comprehensive_firing_rate_array)

# Create combined dataset
comprehensive_combined_data = np.column_stack([
    comprehensive_waveform_scaled,
    comprehensive_firing_rate_scaled
])

print(f"Standardized waveform metrics shape: {comprehensive_waveform_scaled.shape}")
print(f"Standardized firing rate metrics shape: {comprehensive_firing_rate_scaled.shape}")
print(f"Combined comprehensive data shape: {comprehensive_combined_data.shape}")



### 4. CLUSTER
# Apply UMAP dimensionality reduction to comprehensive metrics
print("\n=== APPLYING UMAP TO COMPREHENSIVE METRICS ===")

# UMAP parameters
n_neighbors = 15
min_dist = 0.1
n_components = 2
random_state = 42

# UMAP on comprehensive waveform metrics
print("Applying UMAP to comprehensive waveform metrics...")
umap_waveform_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
waveform_comprehensive_umap = umap_waveform_comprehensive.fit_transform(comprehensive_waveform_scaled)

# UMAP on comprehensive firing rate metrics
print("Applying UMAP to comprehensive firing rate metrics...")
umap_firing_rate_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
firing_rate_comprehensive_umap = umap_firing_rate_comprehensive.fit_transform(comprehensive_firing_rate_scaled)

# UMAP on combined comprehensive data
print("Applying UMAP to combined comprehensive data...")
umap_combined_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
combined_comprehensive_umap = umap_combined_comprehensive.fit_transform(comprehensive_combined_data)

print(f"Comprehensive waveform UMAP shape: {waveform_comprehensive_umap.shape}")
print(f"Comprehensive firing rate UMAP shape: {firing_rate_comprehensive_umap.shape}")
print(f"Comprehensive combined UMAP shape: {combined_comprehensive_umap.shape}")

print("UMAP dimensionality reduction complete!")

In [ ]:
# COMPREHENSIVE CLUSTERING ANALYSIS
print("\n=== COMPREHENSIVE CLUSTERING ANALYSIS ===")

# Method 1: UMAP + K-means Clustering
print("\n--- METHOD 1: UMAP + K-means Clustering ---")

def find_optimal_k_comprehensive(data, k_range, method_name):
    """Find optimal number of clusters using silhouette score"""
    silhouette_scores = []
    inertias = []
    
    print(f"Testing optimal k for {method_name}:")
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(data)
        
        silhouette_avg = silhouette_score(data, labels)
        inertia = kmeans.inertia_
        
        silhouette_scores.append(silhouette_avg)
        inertias.append(inertia)
        
        print(f"k={k}: Silhouette Score = {silhouette_avg:.3f}, Inertia = {inertia:.1f}")
    
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"Optimal number of clusters: {optimal_k}")
    
    return optimal_k, silhouette_scores, inertias

# Test different k values
k_range = range(2, 8)

# Find optimal k for each UMAP embedding
optimal_k_waveform_comp, silhouette_scores_waveform_comp, inertias_waveform_comp = find_optimal_k_comprehensive(
    waveform_comprehensive_umap, k_range, "comprehensive waveform UMAP"
)

optimal_k_firing_rate_comp, silhouette_scores_firing_rate_comp, inertias_firing_rate_comp = find_optimal_k_comprehensive(
    firing_rate_comprehensive_umap, k_range, "comprehensive firing rate UMAP"
)

optimal_k_combined_comp, silhouette_scores_combined_comp, inertias_combined_comp = find_optimal_k_comprehensive(
    combined_comprehensive_umap, k_range, "comprehensive combined UMAP"
)

# Perform final clustering with optimal k values
print("\nPerforming final UMAP + K-means clustering...")

# Waveform clustering
kmeans_waveform_comp_final = KMeans(n_clusters=optimal_k_waveform_comp, random_state=42, n_init=10)
waveform_comp_labels = kmeans_waveform_comp_final.fit_predict(waveform_comprehensive_umap)
print(f"Comprehensive waveform clustering completed with {optimal_k_waveform_comp} clusters")

# Firing rate clustering
kmeans_firing_rate_comp_final = KMeans(n_clusters=optimal_k_firing_rate_comp, random_state=42, n_init=10)
firing_rate_comp_labels = kmeans_firing_rate_comp_final.fit_predict(firing_rate_comprehensive_umap)
print(f"Comprehensive firing rate clustering completed with {optimal_k_firing_rate_comp} clusters")

# Combined clustering
kmeans_combined_comp_final = KMeans(n_clusters=optimal_k_combined_comp, random_state=42, n_init=10)
combined_comp_labels = kmeans_combined_comp_final.fit_predict(combined_comprehensive_umap)
print(f"Comprehensive combined clustering completed with {optimal_k_combined_comp} clusters")

print("UMAP + K-means clustering complete!")



# VISUALIZE UMAP CLUSTERS FOR ALL METHODS WITH COLOR KEYS
print("\n=== UMAP CLUSTER VISUALIZATIONS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

# Create figure with subplots for all UMAP embeddings and clustering results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('UMAP Embeddings and K-means Clustering Results (Comprehensive Analysis)', fontsize=16, fontweight='bold')

# Plot 1: Waveform UMAP with clustering
ax1 = axes[0, 0]
# Use plot instead of scatter for cleaner dots without outlines
for cluster_id in range(optimal_k_waveform_comp):
    mask = waveform_comp_labels == cluster_id
    ax1.plot(waveform_comprehensive_umap[mask, 0], waveform_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_waveform_comp - 1)),
             label=f'Cluster {cluster_id}')
ax1.set_title(f'Waveform UMAP - K-means Clusters (k={optimal_k_waveform_comp})')
ax1.set_xlabel('UMAP 1')
ax1.set_ylabel('UMAP 2')
ax1.grid(True, alpha=0.3)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 2: Firing Rate UMAP with clustering
ax2 = axes[0, 1]
for cluster_id in range(optimal_k_firing_rate_comp):
    mask = firing_rate_comp_labels == cluster_id
    ax2.plot(firing_rate_comprehensive_umap[mask, 0], firing_rate_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_firing_rate_comp - 1)),
             label=f'Cluster {cluster_id}')
ax2.set_title(f'Firing Rate UMAP - K-means Clusters (k={optimal_k_firing_rate_comp})')
ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.grid(True, alpha=0.3)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 3: Combined UMAP with clustering
ax3 = axes[0, 2]
for cluster_id in range(optimal_k_combined_comp):
    mask = combined_comp_labels == cluster_id
    ax3.plot(combined_comprehensive_umap[mask, 0], combined_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_combined_comp - 1)),
             label=f'Cluster {cluster_id}')
ax3.set_title(f'Combined UMAP - K-means Clusters (k={optimal_k_combined_comp})')
ax3.set_xlabel('UMAP 1')
ax3.set_ylabel('UMAP 2')
ax3.grid(True, alpha=0.3)
ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 4: Silhouette scores comparison
ax4 = axes[1, 0]
ax4.plot(k_range, silhouette_scores_waveform_comp, 'o-', label='Waveform UMAP', linewidth=2, markersize=6)
ax4.plot(k_range, silhouette_scores_firing_rate_comp, 's-', label='Firing Rate UMAP', linewidth=2, markersize=6)
ax4.plot(k_range, silhouette_scores_combined_comp, '^-', label='Combined UMAP', linewidth=2, markersize=6)
ax4.set_title('Silhouette Scores vs Number of Clusters')
ax4.set_xlabel('Number of Clusters (k)')
ax4.set_ylabel('Silhouette Score')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Elbow curves comparison
ax5 = axes[1, 1]
ax5.plot(k_range, inertias_waveform_comp, 'o-', label='Waveform UMAP', linewidth=2, markersize=6)
ax5.plot(k_range, inertias_firing_rate_comp, 's-', label='Firing Rate UMAP', linewidth=2, markersize=6)
ax5.plot(k_range, inertias_combined_comp, '^-', label='Combined UMAP', linewidth=2, markersize=6)
ax5.set_title('Elbow Curves - Inertia vs Number of Clusters')
ax5.set_xlabel('Number of Clusters (k)')
ax5.set_ylabel('Inertia')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Cluster size comparison
ax6 = axes[1, 2]
cluster_counts = [optimal_k_waveform_comp, optimal_k_firing_rate_comp, optimal_k_combined_comp]
methods = ['Waveform', 'Firing Rate', 'Combined']
colors = ['skyblue', 'lightcoral', 'lightgreen']

bars = ax6.bar(methods, cluster_counts, color=colors, alpha=0.7, edgecolor='black')
ax6.set_title('Optimal Number of Clusters')
ax6.set_ylabel('Number of Clusters')
ax6.set_ylim(0, max(cluster_counts) + 1)

# Add value labels on bars
for bar, count in zip(bars, cluster_counts):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("cluster_results.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Additional detailed cluster analysis
print("\n=== DETAILED CLUSTER ANALYSIS ===")

def analyze_cluster_characteristics_comprehensive(labels, data_name, n_clusters, waveform_data, firing_rate_data):
    """Analyze characteristics of each cluster"""
    print(f"\n{data_name} Cluster Analysis:")
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_size = np.sum(cluster_mask)
        
        if cluster_size > 0:
            # Analyze waveform characteristics for this cluster
            cluster_waveform_metrics = comprehensive_waveform_array[cluster_mask]
            cluster_firing_rate_metrics = comprehensive_firing_rate_array[cluster_mask]
            
            print(f"  Cluster {cluster_id} (n={cluster_size}):")
            
            # Waveform characteristics
            print(f"    Waveform Metrics:")
            print(f"      Spike Width: {np.mean(cluster_waveform_metrics[:, 0]):.3f} ± {np.std(cluster_waveform_metrics[:, 0]):.3f} ms")
            print(f"      Spike Amplitude: {np.mean(cluster_waveform_metrics[:, 1]):.3f} ± {np.std(cluster_waveform_metrics[:, 1]):.3f}")
            print(f"      Spike Asymmetry: {np.mean(cluster_waveform_metrics[:, 2]):.3f} ± {np.std(cluster_waveform_metrics[:, 2]):.3f}")
            print(f"      Rise Time: {np.mean(cluster_waveform_metrics[:, 3]):.3f} ± {np.std(cluster_waveform_metrics[:, 3]):.3f} ms")
            print(f"      Decay Time: {np.mean(cluster_waveform_metrics[:, 4]):.3f} ± {np.std(cluster_waveform_metrics[:, 4]):.3f} ms")
            
            # Firing rate characteristics
            print(f"    Firing Rate Metrics:")
            print(f"      Firing Rate: {np.mean(cluster_firing_rate_metrics[:, 0]):.2f} ± {np.std(cluster_firing_rate_metrics[:, 0]):.2f} Hz")
            print(f"      Burst Index: {np.mean(cluster_firing_rate_metrics[:, 1]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 1]):.3f}")
            print(f"      ISI CV: {np.mean(cluster_firing_rate_metrics[:, 2]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 2]):.3f}")
            print(f"      ISI Violation Rate: {np.mean(cluster_firing_rate_metrics[:, 3]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 3]):.3f}")
            print(f"      Spike Frequency Adaptation: {np.mean(cluster_firing_rate_metrics[:, 4]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 4]):.3f}")

# Analyze each clustering method
analyze_cluster_characteristics_comprehensive(waveform_comp_labels, "Waveform UMAP", optimal_k_waveform_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)
analyze_cluster_characteristics_comprehensive(firing_rate_comp_labels, "Firing Rate UMAP", optimal_k_firing_rate_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)
analyze_cluster_characteristics_comprehensive(combined_comp_labels, "Combined UMAP", optimal_k_combined_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)

print("\nUMAP cluster visualizations complete!")

In [ ]:
# BAR PLOTS FOR ALL VARIABLES BY CLUSTER WITH STATISTICAL TESTS
print("\n=== VARIABLE ANALYSIS BY CLUSTER ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu

# Define variable names
waveform_vars = ['Spike Width (ms)', 'Spike Amplitude', 'Spike Asymmetry', 'Rise Time (ms)', 'Decay Time (ms)']
firing_rate_vars = ['Firing Rate (Hz)', 'Burst Index', 'ISI CV', 'ISI Violation Rate', 'Spike Frequency Adaptation']
all_vars = waveform_vars + firing_rate_vars

# Get the data arrays
waveform_data = comprehensive_waveform_array  # Shape: (n_neurons, 5)
firing_rate_data = comprehensive_firing_rate_array  # Shape: (n_neurons, 5)
combined_data = np.hstack([waveform_data, firing_rate_data])  # Shape: (n_neurons, 10)

# Get cluster labels
cluster_labels = combined_comp_labels

# Show cluster 0, 1, 2 for plotting,
# and do statistical tests to compare cluster 0 and 2
plot_cluster_ids = [0, 1, 2]
stats_cluster_ids = [0, 2]

# Create figure with subplots for all variables
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Variable Analysis by Cluster (Combined UMAP)', fontsize=16, fontweight='bold')

# Flatten axes for easier indexing
axes_flat = axes.flatten()

# Function to perform statistical tests
def perform_statistical_tests(data_cluster1, data_cluster2, test_name):
    """Perform t-test and Mann-Whitney U test"""
    if len(data_cluster1) == 0 or len(data_cluster2) == 0:
        return None, None
    
    # T-test
    t_stat, t_p = ttest_ind(data_cluster1, data_cluster2)
    
    # Mann-Whitney U test
    u_stat, u_p = mannwhitneyu(data_cluster1, data_cluster2, alternative='two-sided')
    
    return {'t_test': (t_stat, t_p), 'mannwhitney': (u_stat, u_p)}

# Plot each variable
for var_idx in range(10):
    ax = axes_flat[var_idx]
    
    # Get data for this variable
    var_data = combined_data[:, var_idx]
    
    # Calculate means and standard errors for each cluster (0, 1, 2)
    cluster_means = []
    cluster_stds = []
    cluster_sizes = []
    cluster_data = []
    
    for cluster_id in plot_cluster_ids:
        cluster_mask = cluster_labels == cluster_id
        cluster_var_data = var_data[cluster_mask]
        
        if len(cluster_var_data) > 0:
            cluster_means.append(np.mean(cluster_var_data))
            cluster_stds.append(np.std(cluster_var_data))
            cluster_sizes.append(len(cluster_var_data))
            cluster_data.append(cluster_var_data)
        else:
            cluster_means.append(0)
            cluster_stds.append(0)
            cluster_sizes.append(0)
            cluster_data.append(np.array([]))
    
    # Create bar plot
    x_pos = np.arange(len(plot_cluster_ids))
    bars = ax.bar(x_pos, cluster_means, yerr=cluster_stds, capsize=5, 
                  color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7, edgecolor='black')
    
    # Add value labels on bars
    for i, (bar, mean, std, size) in enumerate(zip(bars, cluster_means, cluster_stds, cluster_sizes)):
        if size > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01, 
                   f'{mean:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
    
    # Perform statistical tests between cluster 0 and 2
    idx_0 = plot_cluster_ids.index(0)
    idx_2 = plot_cluster_ids.index(2)
    data_0 = cluster_data[idx_0]
    data_2 = cluster_data[idx_2]

    if len(data_0) > 0 and len(data_2) > 0:
        stats_results = perform_statistical_tests(data_0, data_2, all_vars[var_idx])
        
        if stats_results:
            t_stat, t_p = stats_results['t_test']
            u_stat, u_p = stats_results['mannwhitney']
            
            # Add significance annotation above clusters 0 & 2
            annotation_y = max(cluster_means[idx_0] + cluster_stds[idx_0], 
                               cluster_means[idx_2] + cluster_stds[idx_2]) + 0.05 * (
                max(cluster_means) - min(cluster_means)
            )
            significance = "***" if t_p < 0.001 else "**" if t_p < 0.01 else "*" if t_p < 0.05 else "ns"
            # Draw line and annotation between cluster 0 and 2
            ax.plot([0, 2], [annotation_y, annotation_y], color='k', linewidth=1)
            ax.text(1.0, annotation_y + 0.01, f"T-test p = {t_p:.4f} {significance}", 
                    ha='center', va='bottom', fontweight='bold', fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    # Set plot properties
    ax.set_title(all_vars[var_idx], fontsize=10, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Value')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Cluster {i}' for i in plot_cluster_ids])
    ax.grid(axis='y', alpha=0.3)
    
    # Print detailed statistics for all clusters
    print(f"\n{all_vars[var_idx]}:")
    for idx, cluster_id in enumerate(plot_cluster_ids):
        if cluster_sizes[idx] > 0:
            print(f"  Cluster {cluster_id}: mean = {cluster_means[idx]:.3f} ± {cluster_stds[idx]:.3f} (n={cluster_sizes[idx]})")
    
    # Print statistical test results for 0 vs 2 only
    if len(data_0) > 0 and len(data_2) > 0:
        print(f"  Cluster 0 vs 2: T-test p = {t_p:.4f}, Mann-Whitney p = {u_p:.4f}")

plt.tight_layout()
plt.savefig("summary_stats_clusters.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics table
print("\n=== SUMMARY STATISTICS TABLE ===")
header = "Variable".ljust(25) + "\t" + "\t".join([f"Cluster {i}" for i in plot_cluster_ids])
print(header)
print("-" * (len(header)+20))

for var_idx in range(10):
    var_name = all_vars[var_idx]
    padded_name = var_name.ljust(25)
    
    # Get cluster data for all clusters
    cluster_data_stats = []
    for cluster_id in plot_cluster_ids:
        cluster_mask = cluster_labels == cluster_id
        cluster_var_data = combined_data[cluster_mask, var_idx]
        cluster_data_stats.append(cluster_var_data)
    
    # Format means and standard deviations
    cluster_stats = []
    for idx in range(len(plot_cluster_ids)):
        if len(cluster_data_stats[idx]) > 0:
            mean_val = np.mean(cluster_data_stats[idx])
            std_val = np.std(cluster_data_stats[idx])
            cluster_stats.append(f"{mean_val:.3f} ± {std_val:.3f}")
        else:
            cluster_stats.append("N/A")
    
    print(f"{padded_name}\t" + "\t".join(cluster_stats))

# Statistical significance summary (just for 0 vs 2)
print("\n=== STATISTICAL SIGNIFICANCE SUMMARY (Cluster 0 vs 2) ===")
print("Variable\t\t\tT-test p-value\t\tMann-Whitney p-value")
print("-" * 70)

for var_idx in range(10):
    var_name = all_vars[var_idx]
    padded_name = var_name.ljust(25)
    
    # Get cluster data for 0 and 2 only
    cluster_mask_0 = cluster_labels == 0
    cluster_mask_2 = cluster_labels == 2
    cluster_data_0 = combined_data[cluster_mask_0, var_idx]
    cluster_data_2 = combined_data[cluster_mask_2, var_idx]
    
    if len(cluster_data_0) > 0 and len(cluster_data_2) > 0:
        # T-test
        t_stat, t_p = ttest_ind(cluster_data_0, cluster_data_2)
        
        # Mann-Whitney U test
        u_stat, u_p = mannwhitneyu(cluster_data_0, cluster_data_2, alternative='two-sided')
        
        print(f"{padded_name}\t{t_p:.4f}\t\t\t{u_p:.4f}")
    else:
        print(f"{padded_name}\tN/A\t\t\tN/A")

print("\nVariable analysis complete!")

In [ ]:
# Visualizaiton cell



#  1. DEFINE COMPREHENSIVE VISUALIZATION FUNCTIONS
print("\n=== COMPREHENSIVE VISUALIZATION FUNCTIONS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

def plot_cluster_waveforms_comprehensive(labels, waveforms_array, title_prefix, n_clusters, method_name):
    """Plot average waveforms for each cluster"""
    fig, axes = plt.subplots(1, n_clusters, figsize=(4*n_clusters, 4))
    if n_clusters == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Average Waveforms ({method_name})', fontsize=14, fontweight='bold')
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_waveforms = waveforms_array[cluster_mask]
        
        if len(cluster_waveforms) > 0:
            # Calculate average waveform
            avg_waveform = np.mean(cluster_waveforms, axis=0)
            std_waveform = np.std(cluster_waveforms, axis=0)
            
            # Plot average waveform with error bars
            x_vals = np.arange(len(avg_waveform)) / 30.0  # Convert to ms
            axes[cluster_id].plot(x_vals, avg_waveform, 'b-', linewidth=2, label=f'Cluster {cluster_id}')
            axes[cluster_id].fill_between(x_vals, 
                                        avg_waveform - std_waveform, 
                                        avg_waveform + std_waveform, 
                                        alpha=0.3, color='blue')
            
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_waveforms)})')
            axes[cluster_id].set_xlabel('Time (ms)')
            axes[cluster_id].set_ylabel('Amplitude')
            axes[cluster_id].grid(True, alpha=0.3)
            axes[cluster_id].legend()
    
    plt.tight_layout()
    plt.savefig("cluster_waveforms.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

def plot_region_pie_charts(labels, metadata, title_prefix, n_clusters, method_name):
    """Plot pie charts showing regional distribution for each cluster"""
    fig, axes = plt.subplots(1, n_clusters, figsize=(5*n_clusters, 4))
    if n_clusters == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Regional Distribution ({method_name})', fontsize=14, fontweight='bold')
    
    # Get unique regions
    all_regions = [meta['region'] for meta in metadata]
    unique_regions = list(set(all_regions))
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_metadata = [metadata[i] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_metadata) > 0:
            # Count regions in this cluster
            cluster_regions = [meta['region'] for meta in cluster_metadata]
            region_counts = {region: cluster_regions.count(region) for region in unique_regions}
            
            # Create pie chart
            labels_pie = list(region_counts.keys())
            sizes = list(region_counts.values())
            colors = plt.cm.Set3(np.linspace(0, 1, len(labels_pie)))
            
            axes[cluster_id].pie(sizes, labels=labels_pie, autopct='%1.1f%%', colors=colors)
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_metadata)})')
    
    plt.tight_layout()
    plt.savefig("cluster_region.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

print("Comprehensive visualization functions defined!")



# 2. COMPREHENSIVE VISUALIZATIONS FOR UMAP + K-MEANS CLUSTERING
print("\n=== COMPREHENSIVE VISUALIZATIONS: UMAP + K-MEANS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

# Convert waveforms_flat to numpy array for visualization
waveforms_array_for_viz = np.array(waveforms_flat)

# Combined UMAP + K-means visualizations
print("\n--- Combined UMAP + K-means Clustering ---")
plot_cluster_waveforms_comprehensive(combined_comp_labels, waveforms_array_for_viz, 
                                    "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
plot_region_pie_charts(combined_comp_labels, comprehensive_metadata, 
                      "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_depth_histograms(combined_comp_labels, comprehensive_metadata, 
#                     "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_subgroup_bar_charts(combined_comp_labels, comprehensive_metadata, 
#                        "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")


In [ ]:
# ET sparse specific visualizations
print("\n=== ET-DENSE vs ET-SPARSE REGIONAL ANALYSIS ===")



# 0. Define ET-dense and ET-sparse regions
et_dense_regions = ['vPrCG', 'PoCG']  # Motor and somatosensory cortex
et_sparse_regions = ['pSTG', 'SFG', 'aSTG', 'aMTG', 'MFG', 'parsOp', 'parsTr', 'parsOr', 'SMG']

print(f"ET-dense regions: {et_dense_regions}")
print(f"ET-sparse regions: {et_sparse_regions}")

# Create masks for ET-dense and ET-sparse regions
et_dense_mask = np.array([neuron['region'] in et_dense_regions for neuron in comprehensive_metadata])
et_sparse_mask = np.array([neuron['region'] in et_sparse_regions for neuron in comprehensive_metadata])

print(f"ET-dense neurons: {np.sum(et_dense_mask)}")
print(f"ET-sparse neurons: {np.sum(et_sparse_mask)}")

# Separate the combined clustering data
et_dense_labels = combined_comp_labels[et_dense_mask]
et_sparse_labels = combined_comp_labels[et_sparse_mask]

et_dense_metadata = [comprehensive_metadata[i] for i in range(len(comprehensive_metadata)) if et_dense_mask[i]]
et_sparse_metadata = [comprehensive_metadata[i] for i in range(len(comprehensive_metadata)) if et_sparse_mask[i]]

et_dense_waveforms = waveforms_array_for_viz[et_dense_mask]
et_sparse_waveforms = waveforms_array_for_viz[et_sparse_mask]



# 1. ET-DENSE REGIONS VISUALIZATION
print("\n=== ET-DENSE REGIONS VISUALIZATION ===")
print(f"Analyzing {len(et_dense_labels)} neurons from ET-dense regions (vPrCG, PoCG)")

# 1. Waveform visualizations for ET-dense
print("\n--- ET-Dense: Waveform Analysis ---")
plot_cluster_waveforms_comprehensive(et_dense_labels, et_dense_waveforms, 
                                    "ET-Dense Combined UMAP", optimal_k_combined_comp, "ET-Dense")

# 2. Region pie charts for ET-dense
print("\n--- ET-Dense: Region Distribution ---")
plot_region_pie_charts(et_dense_labels, et_dense_metadata, 
                      "ET-Dense Combined UMAP", optimal_k_combined_comp, "ET-Dense")



# 2. ET-SPARSE REGIONS VISUALIZATION
print("\n=== ET-SPARSE REGIONS VISUALIZATION ===")
print(f"Analyzing {len(et_sparse_labels)} neurons from ET-sparse regions (all others)")

# 1. Waveform visualizations for ET-sparse
print("\n--- ET-Sparse: Waveform Analysis ---")
plot_cluster_waveforms_comprehensive(et_sparse_labels, et_sparse_waveforms, 
                                    "ET-Sparse Combined UMAP", optimal_k_combined_comp, "ET-Sparse")

# 2. Region pie charts for ET-sparse
print("\n--- ET-Sparse: Region Distribution ---")
plot_region_pie_charts(et_sparse_labels, et_sparse_metadata, 
                      "ET-Sparse Combined UMAP", optimal_k_combined_comp, "ET-Sparse")

In [ ]:
# PIE CHARTS FOR IDH MUTATION STATUS AND GRADE
# SEPARATE ANALYSES FOR OPERCULAR = 0 AND OPERCULAR = 1
plt.rcParams['svg.fonttype'] = 'none' 

# Filter metadata by opercular status
et_sparse_metadata_op0 = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 0]
et_sparse_metadata_op1 = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 1]

# Function to calculate counts and percentages
def calculate_stats(metadata_subset):
    # IDH mutation status counts
    idh_mut_count = sum(1 for neuron in metadata_subset if neuron['pathology'] in ['ast', 'oli'])
    idh_wt_count = sum(1 for neuron in metadata_subset if neuron['pathology'] == 'gbm')
    idh_total = idh_mut_count + idh_wt_count
    
    # Grade counts
    grade_2_3_count = sum(1 for neuron in metadata_subset if neuron['grade'] in [2, 3])
    grade_4_count = sum(1 for neuron in metadata_subset if neuron['grade'] == 4)
    grade_total = grade_2_3_count + grade_4_count
    
    # Calculate percentages
    idh_mut_pct = (idh_mut_count / idh_total * 100) if idh_total > 0 else 0
    idh_wt_pct = (idh_wt_count / idh_total * 100) if idh_total > 0 else 0
    
    grade_2_3_pct = (grade_2_3_count / grade_total * 100) if grade_total > 0 else 0
    grade_4_pct = (grade_4_count / grade_total * 100) if grade_total > 0 else 0
    
    return {
        'idh_mut_count': idh_mut_count,
        'idh_wt_count': idh_wt_count,
        'idh_total': idh_total,
        'idh_mut_pct': idh_mut_pct,
        'idh_wt_pct': idh_wt_pct,
        'grade_2_3_count': grade_2_3_count,
        'grade_4_count': grade_4_count,
        'grade_total': grade_total,
        'grade_2_3_pct': grade_2_3_pct,
        'grade_4_pct': grade_4_pct
    }

# Calculate stats for both groups
stats_op0 = calculate_stats(et_sparse_metadata_op0)
stats_op1 = calculate_stats(et_sparse_metadata_op1)

# Create figure with 4 subplots (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('ET-Sparse Regions: IDH Status and Grade Distribution by Opercular Status', 
             fontsize=16, fontweight='bold', y=0.995)

# OPERCULAR = 0: IDH Mutation Status
ax1 = axes[0, 0]
idh_labels = ['IDH mut', 'IDH wt']
idh_counts_op0 = [stats_op0['idh_mut_count'], stats_op0['idh_wt_count']]
idh_colors = ['#2ca02c', '#ff7f0e']  # Green for mut, Orange for wt

wedges, texts, autotexts = ax1.pie(idh_counts_op0, labels=idh_labels, colors=idh_colors, autopct='', 
                                   startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, idh_counts_op0, 
                                             [stats_op0['idh_mut_pct'], stats_op0['idh_wt_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax1.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax1.set_title('Opercular = 0: IDH Mutation Status', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 0: Grade Distribution
ax2 = axes[0, 1]
grade_labels = ['Grade 2/3', 'Grade 4']
grade_counts_op0 = [stats_op0['grade_2_3_count'], stats_op0['grade_4_count']]
grade_colors = ['#1f77b4', '#d62728']  # Blue for 2/3, Red for 4

wedges, texts, autotexts = ax2.pie(grade_counts_op0, labels=grade_labels, colors=grade_colors, autopct='', 
                                    startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, grade_counts_op0, 
                                             [stats_op0['grade_2_3_pct'], stats_op0['grade_4_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax2.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax2.set_title('Opercular = 0: Grade Distribution', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 1: IDH Mutation Status
ax3 = axes[1, 0]
idh_counts_op1 = [stats_op1['idh_mut_count'], stats_op1['idh_wt_count']]

wedges, texts, autotexts = ax3.pie(idh_counts_op1, labels=idh_labels, colors=idh_colors, autopct='', 
                                   startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, idh_counts_op1, 
                                             [stats_op1['idh_mut_pct'], stats_op1['idh_wt_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax3.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax3.set_title('Opercular = 1: IDH Mutation Status', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 1: Grade Distribution
ax4 = axes[1, 1]
grade_counts_op1 = [stats_op1['grade_2_3_count'], stats_op1['grade_4_count']]

wedges, texts, autotexts = ax4.pie(grade_counts_op1, labels=grade_labels, colors=grade_colors, autopct='', 
                                    startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, grade_counts_op1, 
                                             [stats_op1['grade_2_3_pct'], stats_op1['grade_4_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax4.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax4.set_title('Opercular = 1: Grade Distribution', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig("idh_grade_pie_charts_by_opercular.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("=== OPERCULAR = 0 ===")
print("IDH Mutation Status:")
print(f"  IDH mut: {stats_op0['idh_mut_count']} ({stats_op0['idh_mut_pct']:.1f}%)")
print(f"  IDH wt: {stats_op0['idh_wt_count']} ({stats_op0['idh_wt_pct']:.1f}%)")
print(f"  Total: {stats_op0['idh_total']}")
print("\nGrade Distribution:")
print(f"  Grade 2/3: {stats_op0['grade_2_3_count']} ({stats_op0['grade_2_3_pct']:.1f}%)")
print(f"  Grade 4: {stats_op0['grade_4_count']} ({stats_op0['grade_4_pct']:.1f}%)")
print(f"  Total: {stats_op0['grade_total']}")

print("\n=== OPERCULAR = 1 ===")
print("IDH Mutation Status:")
print(f"  IDH mut: {stats_op1['idh_mut_count']} ({stats_op1['idh_mut_pct']:.1f}%)")
print(f"  IDH wt: {stats_op1['idh_wt_count']} ({stats_op1['idh_wt_pct']:.1f}%)")
print(f"  Total: {stats_op1['idh_total']}")
print("\nGrade Distribution:")
print(f"  Grade 2/3: {stats_op1['grade_2_3_count']} ({stats_op1['grade_2_3_pct']:.1f}%)")
print(f"  Grade 4: {stats_op1['grade_4_count']} ({stats_op1['grade_4_pct']:.1f}%)")
print(f"  Total: {stats_op1['grade_total']}")

In [ ]:
# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 0 ONLY
# WITH STATISTICAL COMPARISONS AND POST-HOC PAIRWISE TESTS
plt.rcParams['svg.fonttype'] = 'none' 

# Function to calculate proportions for opercular subgroups
def calculate_proportions_opercular(labels, metadata, group_variable, group_values, opercular_value):
    """Calculate proportion of each cluster within a specific group and opercular status"""
    if isinstance(group_values, str):
        group_values = [group_values]
    
    group_mask = np.array([neuron[group_variable] in group_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    combined_mask = group_mask & opercular_mask
    
    group_labels = labels[combined_mask]
    
    if len(group_labels) == 0:
        return np.zeros(3)  # Return zeros if no data
    
    cluster_counts = np.bincount(group_labels, minlength=3)
    proportions = cluster_counts / np.sum(cluster_counts)
    return proportions

# Function to perform chi-square test for cluster proportions
def chi_square_test_clusters(labels, metadata, group_variable, group1_values, group2_values, opercular_value):
    """Perform chi-square test comparing cluster distributions between two groups"""
    from scipy.stats import chi2_contingency
    
    # Get masks for both groups
    group1_mask = np.array([neuron[group_variable] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron[group_variable] in group2_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    
    combined_mask1 = group1_mask & opercular_mask
    combined_mask2 = group2_mask & opercular_mask
    
    group1_labels = labels[combined_mask1]
    group2_labels = labels[combined_mask2]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None, None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    contingency_table = np.array([group1_counts, group2_counts])
    
    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    return chi2, p_value

# Function to perform post-hoc pairwise comparisons for individual clusters
def posthoc_cluster_comparisons(labels, metadata, group_variable, group1_values, group2_values, opercular_value):
    """Perform post-hoc comparisons for each individual cluster"""
    from scipy.stats import chi2_contingency
    
    # Get masks for both groups
    group1_mask = np.array([neuron[group_variable] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron[group_variable] in group2_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    
    combined_mask1 = group1_mask & opercular_mask
    combined_mask2 = group2_mask & opercular_mask
    
    group1_labels = labels[combined_mask1]
    group2_labels = labels[combined_mask2]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    results = []
    
    # Test each cluster individually (cluster vs all others)
    for cluster_id in range(3):
        # Create 2x2 contingency table: cluster vs all others
        cluster1_count = group1_counts[cluster_id]
        other1_count = np.sum(group1_counts) - cluster1_count
        
        cluster2_count = group2_counts[cluster_id]
        other2_count = np.sum(group2_counts) - cluster2_count
        
        contingency_2x2 = np.array([[cluster1_count, other1_count],
                                   [cluster2_count, other2_count]])
        
        # Perform chi-square test
        chi2, p_value, dof, expected = chi2_contingency(contingency_2x2)
        
        results.append({
            'cluster': cluster_id,
            'chi2': chi2,
            'p_value': p_value,
            'group1_prop': cluster1_count / np.sum(group1_counts),
            'group2_prop': cluster2_count / np.sum(group2_counts)
        })
    
    return results

# Define colors for clusters
cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green

# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 0 ONLY
print("ET-Sparse regions analysis by opercular status = 0:")

# ET-Sparse: Pathology proportions by opercular status = 0
et_sparse_oli_ast_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], 0)
et_sparse_gbm_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['gbm'], 0)

# ET-Sparse: Grade proportions by opercular status = 0
et_sparse_grade2_3_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], 0)
et_sparse_grade4_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [4], 0)

print(f"ET-Sparse oli+ast opercular=0 proportions: {et_sparse_oli_ast_op0_props}")
print(f"ET-Sparse gbm opercular=0 proportions: {et_sparse_gbm_op0_props}")
print(f"ET-Sparse grade 2+3 opercular=0 proportions: {et_sparse_grade2_3_op0_props}")
print(f"ET-Sparse grade 4 opercular=0 proportions: {et_sparse_grade4_op0_props}")

# Perform statistical tests
print("\n=== STATISTICAL TESTS ===")

# Pathology comparison (oli+ast vs gbm)
chi2_path, p_path = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 0)
print(f"Pathology comparison (oli+ast vs gbm):")
print(f"  Overall Chi-square = {chi2_path:.4f}, p-value = {p_path:.4f}")

# Post-hoc pairwise comparisons for pathology
path_posthoc = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 0)
if path_posthoc:
    print("  Post-hoc pairwise comparisons:")
    for result in path_posthoc:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      oli+ast prop = {result['group1_prop']:.3f}, gbm prop = {result['group2_prop']:.3f}")

# Grade comparison (2+3 vs 4)
chi2_grade, p_grade = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 0)
print(f"\nGrade comparison (2+3 vs 4):")
print(f"  Overall Chi-square = {chi2_grade:.4f}, p-value = {p_grade:.4f}")

# Post-hoc pairwise comparisons for grade
grade_posthoc = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 0)
if grade_posthoc:
    print("  Post-hoc pairwise comparisons:")
    for result in grade_posthoc:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      grade 2+3 prop = {result['group1_prop']:.3f}, grade 4 prop = {result['group2_prop']:.3f}")

# Create plots for ET-Sparse regions by opercular status = 0 only
fig_sparse, axes_sparse = plt.subplots(1, 2, figsize=(12, 6))
fig_sparse.suptitle('ET-Sparse Regions: Cluster Proportions (Opercular=0)', fontsize=16, fontweight='bold')

# PLOT 1: ET-Sparse Pathology Proportions - Opercular=0
ax1 = axes_sparse[0]
groups = ['oli+ast', 'gbm']
et_sparse_path_op0_data = [et_sparse_oli_ast_op0_props, et_sparse_gbm_op0_props]

# Create stacked bars
x_pos = np.arange(len(groups))
width = 0.6

bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_path_op0_data).T, cluster_colors)):
    ax1.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax1.set_title('ET-Sparse: Pathology Proportions (Opercular=0)')
ax1.set_ylabel('Proportion')
ax1.set_ylim(0, 1)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(groups)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_path_op0_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax1.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_path is not None:
    significance = "***" if p_path < 0.001 else "**" if p_path < 0.01 else "*" if p_path < 0.05 else "ns"
    ax1.text(0.5, 1.05, f"Overall p = {p_path:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax1.transAxes)

# PLOT 2: ET-Sparse Grade Proportions - Opercular=0
ax2 = axes_sparse[1]
groups = ['Grade 2+3', 'Grade 4']
et_sparse_grade_op0_data = [et_sparse_grade2_3_op0_props, et_sparse_grade4_op0_props]

# Create stacked bars
bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_grade_op0_data).T, cluster_colors)):
    ax2.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax2.set_title('ET-Sparse: Grade Proportions (Opercular=0)')
ax2.set_ylabel('Proportion')
ax2.set_ylim(0, 1)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(groups)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_grade_op0_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax2.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_grade is not None:
    significance = "***" if p_grade < 0.001 else "**" if p_grade < 0.01 else "*" if p_grade < 0.05 else "ns"
    ax2.text(0.5, 1.05, f"Overall p = {p_grade:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax2.transAxes)

plt.tight_layout()
plt.savefig("groupproprotions_sparse.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print detailed cluster-by-cluster comparisons
print("\n=== DETAILED CLUSTER COMPARISONS ===")

# Pathology comparison - cluster by cluster
print("Pathology comparison (oli+ast vs gbm) - Cluster proportions:")
for i in range(3):
    oli_ast_prop = et_sparse_oli_ast_op0_props[i]
    gbm_prop = et_sparse_gbm_op0_props[i]
    diff = gbm_prop - oli_ast_prop


In [ ]:
# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 1 ONLY
# WITH STATISTICAL COMPARISONS AND POST-HOC PAIRWISE TESTS
plt.rcParams['svg.fonttype'] = 'none' 

# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 1 ONLY
print("ET-Sparse regions analysis by opercular status = 1:")

# ET-Sparse: Pathology proportions by opercular status = 1
et_sparse_oli_ast_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], 1)
et_sparse_gbm_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['gbm'], 1)

# ET-Sparse: Grade proportions by opercular status = 1
et_sparse_grade2_3_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], 1)
et_sparse_grade4_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [4], 1)

print(f"ET-Sparse oli+ast opercular=1 proportions: {et_sparse_oli_ast_op1_props}")
print(f"ET-Sparse gbm opercular=1 proportions: {et_sparse_gbm_op1_props}")
print(f"ET-Sparse grade 2+3 opercular=1 proportions: {et_sparse_grade2_3_op1_props}")
print(f"ET-Sparse grade 4 opercular=1 proportions: {et_sparse_grade4_op1_props}")

# Perform statistical tests
print("\n=== STATISTICAL TESTS ===")

# Pathology comparison (oli+ast vs gbm)
chi2_path_op1, p_path_op1 = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 1)
print(f"Pathology comparison (oli+ast vs gbm):")
print(f"  Overall Chi-square = {chi2_path_op1:.4f}, p-value = {p_path_op1:.4f}")

# Post-hoc pairwise comparisons for pathology
path_posthoc_op1 = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 1)
if path_posthoc_op1:
    print("  Post-hoc pairwise comparisons:")
    for result in path_posthoc_op1:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      oli+ast prop = {result['group1_prop']:.3f}, gbm prop = {result['group2_prop']:.3f}")

# Grade comparison (2+3 vs 4)
chi2_grade_op1, p_grade_op1 = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 1)
print(f"\nGrade comparison (2+3 vs 4):")
print(f"  Overall Chi-square = {chi2_grade_op1:.4f}, p-value = {p_grade_op1:.4f}")

# Post-hoc pairwise comparisons for grade
grade_posthoc_op1 = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 1)
if grade_posthoc_op1:
    print("  Post-hoc pairwise comparisons:")
    for result in grade_posthoc_op1:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      grade 2+3 prop = {result['group1_prop']:.3f}, grade 4 prop = {result['group2_prop']:.3f}")

# Create plots for ET-Sparse regions by opercular status = 1 only
fig_sparse_op1, axes_sparse_op1 = plt.subplots(1, 2, figsize=(12, 6))
fig_sparse_op1.suptitle('ET-Sparse Regions: Cluster Proportions (Opercular=1)', fontsize=16, fontweight='bold')

# PLOT 1: ET-Sparse Pathology Proportions - Opercular=1
ax1 = axes_sparse_op1[0]
groups = ['oli+ast', 'gbm']
et_sparse_path_op1_data = [et_sparse_oli_ast_op1_props, et_sparse_gbm_op1_props]

# Create stacked bars
x_pos = np.arange(len(groups))
width = 0.6

bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_path_op1_data).T, cluster_colors)):
    ax1.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax1.set_title('ET-Sparse: Pathology Proportions (Opercular=1)')
ax1.set_ylabel('Proportion')
ax1.set_ylim(0, 1)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(groups)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_path_op1_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax1.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_path_op1 is not None:
    significance = "***" if p_path_op1 < 0.001 else "**" if p_path_op1 < 0.01 else "*" if p_path_op1 < 0.05 else "ns"
    ax1.text(0.5, 1.05, f"Overall p = {p_path_op1:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax1.transAxes)

# PLOT 2: ET-Sparse Grade Proportions - Opercular=1
ax2 = axes_sparse_op1[1]
groups = ['Grade 2+3', 'Grade 4']
et_sparse_grade_op1_data = [et_sparse_grade2_3_op1_props, et_sparse_grade4_op1_props]

# Create stacked bars
bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_grade_op1_data).T, cluster_colors)):
    ax2.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax2.set_title('ET-Sparse: Grade Proportions (Opercular=1)')
ax2.set_ylabel('Proportion')
ax2.set_ylim(0, 1)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(groups)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_grade_op1_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax2.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_grade_op1 is not None:
    significance = "***" if p_grade_op1 < 0.001 else "**" if p_grade_op1 < 0.01 else "*" if p_grade_op1 < 0.05 else "ns"
    ax2.text(0.5, 1.05, f"Overall p = {p_grade_op1:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax2.transAxes)

plt.tight_layout()
plt.savefig("groupproprotions_sparse_op1.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print detailed cluster-by-cluster comparisons
print("\n=== DETAILED CLUSTER COMPARISONS ===")

# Pathology comparison - cluster by cluster
print("Pathology comparison (oli+ast vs gbm) - Cluster proportions:")
for i in range(3):
    oli_ast_prop = et_sparse_oli_ast_op1_props[i]
    gbm_prop = et_sparse_gbm_op1_props[i]
    diff = gbm_prop - oli_ast_prop
    print(f"  Cluster {i}: oli+ast = {oli_ast_prop:.3f}, gbm = {gbm_prop:.3f}, diff = {diff:.3f}")

# Grade comparison - cluster by cluster
print("\nGrade comparison (2+3 vs 4) - Cluster proportions:")
for i in range(3):
    grade2_3_prop = et_sparse_grade2_3_op1_props[i]
    grade4_prop = et_sparse_grade4_op1_props[i]
    diff = grade4_prop - grade2_3_prop
    print(f"  Cluster {i}: grade 2+3 = {grade2_3_prop:.3f}, grade 4 = {grade4_prop:.3f}, diff = {diff:.3f}")

In [ ]:
# DEPTH ANALYSIS: ET DENSE (vPrCG) vs ET SPARSE (ALL OTHERS) BY CLUSTER
# Compare proportion of neurons in depth bins for each cluster
# Test each bin independently with chi-square, then FDR correct
# Shows BOTH within-cluster and across-all-clusters FDR correction
plt.rcParams['svg.fonttype'] = 'none' 

from scipy.stats import chi2_contingency, ks_2samp
from statsmodels.stats.multitest import multipletests

def create_depth_bins(depths, n_bins=6):
    """Create depth bins from 0 to max depth"""
    depths_clean = np.maximum(depths, 0)
    max_depth = np.max(depths_clean)
    bin_edges = np.linspace(0, max_depth, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    return bin_edges, bin_centers

def get_depth_bin_counts(neurons, bin_edges):
    """Calculate depth bin counts for a group of neurons"""
    depths = [n['depth'] for n in neurons]
    depths_clean = np.maximum(depths, 0)
    counts, _ = np.histogram(depths_clean, bins=bin_edges)
    return counts.astype(float)

def get_depth_bin_proportions(neurons, bin_edges):
    """Calculate depth bin proportions for a group of neurons"""
    counts = get_depth_bin_counts(neurons, bin_edges)
    total = np.sum(counts)
    if total == 0:
        return np.zeros_like(counts)
    return counts / total

# Get all neurons with their metadata and cluster labels
all_neurons = []
for i, neuron_meta in enumerate(comprehensive_metadata):
    neuron_data = neuron_meta.copy()
    neuron_data['cluster_label'] = combined_comp_labels[i]
    
    # Determine ET status: ET-dense = vPrCG, ET-sparse = all others
    region = neuron_data['region']
    neuron_data['et_status'] = 'ET-dense' if region == 'vPrCG' else 'ET-sparse'
    
    all_neurons.append(neuron_data)

# Create depth bins (6 equally spaced)
depths = [neuron['depth'] for neuron in all_neurons]
bin_edges, bin_centers = create_depth_bins(depths, n_bins=6)

# Separate neurons by ET status
et_dense_neurons = [n for n in all_neurons if n['et_status'] == 'ET-dense']
et_sparse_neurons = [n for n in all_neurons if n['et_status'] == 'ET-sparse']

print("=" * 80)
print("DEPTH ANALYSIS: ET DENSE (vPrCG) vs ET SPARSE (ALL OTHERS) BY CLUSTER")
print("=" * 80)
print(f"Total neurons: {len(all_neurons)}")
print(f"ET-dense (vPrCG): {len(et_dense_neurons)} neurons")
print(f"ET-sparse (all others): {len(et_sparse_neurons)} neurons")

# Calculate proportions and perform chi-square tests for each cluster
cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
results_by_cluster = {}
all_p_values = []  # Collect all p-values for across-all-clusters FDR correction
all_p_indices = []  # Track (cluster_id, bin_idx) for each p-value

# Store KS test results for printing later
ks_test_results = {}

for cluster_id in range(3):
    # Get neurons for this cluster
    et_dense_cluster = [n for n in et_dense_neurons if n['cluster_label'] == cluster_id]
    et_sparse_cluster = [n for n in et_sparse_neurons if n['cluster_label'] == cluster_id]
    
    # Calculate depth bin counts
    et_dense_counts = get_depth_bin_counts(et_dense_cluster, bin_edges)
    et_sparse_counts = get_depth_bin_counts(et_sparse_cluster, bin_edges)
    
    # Calculate proportions
    et_dense_props = get_depth_bin_proportions(et_dense_cluster, bin_edges)
    et_sparse_props = get_depth_bin_proportions(et_sparse_cluster, bin_edges)
    
    # KS test (using raw depths, not binned props)
    dense_depths = [n['depth'] for n in et_dense_cluster]
    sparse_depths = [n['depth'] for n in et_sparse_cluster]
    if len(dense_depths) > 0 and len(sparse_depths) > 0:
        ks_stat, ks_pvalue = ks_2samp(dense_depths, sparse_depths)
    else:
        ks_stat, ks_pvalue = np.nan, np.nan
    ks_test_results[cluster_id] = (ks_stat, ks_pvalue)
    
    # Test each bin independently with chi-square (2x2 contingency: bin vs all other bins)
    bin_p_values = []
    bin_chi2_stats = []
    bin_dofs = []
    
    for bin_idx in range(len(bin_centers)):
        # Create 2x2 contingency table for this bin
        # Row 1: ET-dense: [in this bin, in other bins]
        # Row 2: ET-sparse: [in this bin, in other bins]
        et_dense_in_bin = et_dense_counts[bin_idx]
        et_dense_other = np.sum(et_dense_counts) - et_dense_in_bin
        
        et_sparse_in_bin = et_sparse_counts[bin_idx]
        et_sparse_other = np.sum(et_sparse_counts) - et_sparse_in_bin
        
        contingency_2x2 = np.array([[et_dense_in_bin, et_dense_other],
                                   [et_sparse_in_bin, et_sparse_other]])
        
        # Perform chi-square test
        try:
            chi2_stat, p_value, dof, expected = chi2_contingency(contingency_2x2)
            bin_chi2_stats.append(chi2_stat)
            bin_p_values.append(p_value)
            bin_dofs.append(dof)
            
            # Store for across-all-clusters FDR correction
            all_p_values.append(p_value)
            all_p_indices.append((cluster_id, bin_idx))
        except Exception as e:
            bin_chi2_stats.append(np.nan)
            bin_p_values.append(np.nan)
            bin_dofs.append(1)
            all_p_values.append(np.nan)
            all_p_indices.append((cluster_id, bin_idx))
    
    # FDR correction WITHIN this cluster (6 bins)
    bin_p_values_array = np.array(bin_p_values)
    valid_mask = ~np.isnan(bin_p_values_array)
    
    bin_p_corrected_within = np.full(len(bin_p_values), np.nan)
    bin_rejected_within = np.full(len(bin_p_values), False)
    
    if np.sum(valid_mask) > 0:
        rejected, p_corrected, _, _ = multipletests(
            bin_p_values_array[valid_mask], 
            alpha=0.05, 
            method='fdr_bh'
        )
        bin_p_corrected_within[valid_mask] = p_corrected
        bin_rejected_within[valid_mask] = rejected
    
    results_by_cluster[cluster_id] = {
        'et_dense_neurons': et_dense_cluster,
        'et_sparse_neurons': et_sparse_cluster,
        'et_dense_props': et_dense_props,
        'et_sparse_props': et_sparse_props,
        'et_dense_counts': et_dense_counts,
        'et_sparse_counts': et_sparse_counts,
        'bin_p_values': bin_p_values,
        'bin_p_corrected_within': bin_p_corrected_within,
        'bin_rejected_within': bin_rejected_within,
        'bin_chi2_stats': bin_chi2_stats
    }

# FDR correction ACROSS ALL clusters and bins (18 tests total: 3 clusters × 6 bins)
all_p_values_array = np.array(all_p_values)
valid_mask_all = ~np.isnan(all_p_values_array)

all_p_corrected_across = np.full(len(all_p_values), np.nan)
all_rejected_across = np.full(len(all_p_values), False)

if np.sum(valid_mask_all) > 0:
    rejected_all, p_corrected_all, _, _ = multipletests(
        all_p_values_array[valid_mask_all],
        alpha=0.05,
        method='fdr_bh'
    )
    all_p_corrected_across[valid_mask_all] = p_corrected_all
    all_rejected_across[valid_mask_all] = rejected_all

# Update results with across-all-clusters FDR correction
for idx, (cluster_id, bin_idx) in enumerate(all_p_indices):
    if 'bin_p_corrected_across' not in results_by_cluster[cluster_id]:
        results_by_cluster[cluster_id]['bin_p_corrected_across'] = np.full(len(bin_centers), np.nan)
        results_by_cluster[cluster_id]['bin_rejected_across'] = np.full(len(bin_centers), False)
    
    results_by_cluster[cluster_id]['bin_p_corrected_across'][bin_idx] = all_p_corrected_across[idx]
    results_by_cluster[cluster_id]['bin_rejected_across'][bin_idx] = all_rejected_across[idx]

print("\n" + "=" * 80)
print("FDR CORRECTION COMPARISON")
print("=" * 80)
for cluster_id in range(3):
    result = results_by_cluster[cluster_id]
    n_sig_within = np.sum(result['bin_rejected_within'])
    n_sig_across = np.sum(result['bin_rejected_across'])
    print(f"Cluster {cluster_id}:")
    print(f"  Within-cluster FDR: {n_sig_within}/{len(bin_centers)} bins significant")
    print(f"  Across-all-clusters FDR: {n_sig_across}/{len(bin_centers)} bins significant")

print("\n" + "=" * 80)
print("KS TEST: ET-dense vs ET-sparse Depth Distributions By Cluster")
print("=" * 80)
for cluster_id in range(3):
    ks_stat, ks_pval = ks_test_results[cluster_id]
    ndense = len(results_by_cluster[cluster_id]['et_dense_neurons'])
    nsparse = len(results_by_cluster[cluster_id]['et_sparse_neurons'])
    if np.isnan(ks_stat) or np.isnan(ks_pval):
        print(f"Cluster {cluster_id}: Not enough data (n_dense={ndense}, n_sparse={nsparse}) for KS test.")
    else:
        print(f"Cluster {cluster_id}: KS statistic = {ks_stat:.4f}, p = {ks_pval:.4g} (n_dense={ndense}, n_sparse={nsparse})")

# Create visualization - one plot per cluster
fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)
fig.suptitle('Depth Distribution: ET Dense (vPrCG) vs ET Sparse by Cluster\n(FDR-corrected: within-cluster AND across-all-clusters)', 
             fontsize=16, fontweight='bold', y=1.02)

for cluster_id in range(3):
    ax = fig.add_subplot(gs[0, cluster_id])
    
    result = results_by_cluster[cluster_id]
    et_dense_props = result['et_dense_props']
    et_sparse_props = result['et_sparse_props']
    
    # Use across-all-clusters FDR correction (more conservative)
    bin_p_corrected = result['bin_p_corrected_across']
    bin_rejected = result['bin_rejected_across']
    
    # Also show within-cluster for comparison
    bin_rejected_within = result['bin_rejected_within']
    
    # Plot proportions with lines and bars
    ax.plot(bin_centers, et_dense_props, 'o-', label='ET-dense (vPrCG)', 
            color='tab:blue', linewidth=2, markersize=6)
    ax.plot(bin_centers, et_sparse_props, 'o-', label='ET-sparse (others)', 
            color='tab:orange', linewidth=2, markersize=6)
    
    # Add bar chart
    width = (bin_edges[1] - bin_edges[0]) * 0.35
    
    for i in range(len(bin_centers)):
        # Color based on across-all-clusters significance (more conservative)
        bar_color_dense = 'tab:blue'
        bar_color_sparse = 'tab:orange'
        if bin_rejected[i]:
            bar_color_dense = '#0000CD'  # Darker blue
            bar_color_sparse = '#FF4500'  # Darker orange
        
        ax.bar(bin_centers[i] - width/2, et_dense_props[i], width, 
               color=bar_color_dense, alpha=0.5, label='_nolegend_')
        ax.bar(bin_centers[i] + width/2, et_sparse_props[i], width,
               color=bar_color_sparse, alpha=0.5, label='_nolegend_')
        
        # Add significance markers
        # * = significant in across-all-clusters FDR
        # + = significant only in within-cluster FDR
        if bin_rejected[i]:
            max_height = max(et_dense_props[i], et_sparse_props[i])
            ax.plot(bin_centers[i], max_height * 1.1, '*', 
                   color='black', markersize=12, markeredgewidth=1, 
                   zorder=10, label='_nolegend_')
        elif bin_rejected_within[i]:
            max_height = max(et_dense_props[i], et_sparse_props[i])
            ax.plot(bin_centers[i], max_height * 1.1, '+', 
                   color='gray', markersize=10, markeredgewidth=2, 
                   zorder=10, label='_nolegend_')
    
    # Add statistics annotation
    n_sig_across = np.sum(bin_rejected)
    n_sig_within = np.sum(bin_rejected_within)
    
    stats_text = f"FDR correction:\n"
    stats_text += f"Across-all: {n_sig_across}/{len(bin_centers)} sig\n"
    stats_text += f"Within-cluster: {n_sig_within}/{len(bin_centers)} sig"
    
    # Show p-values for significant bins (using across-all-clusters)
    sig_bins = np.where(bin_rejected)[0]
    if len(sig_bins) > 0:
        stats_text += "\n\nSignificant (across-all):"
        for bin_idx in sig_bins[:3]:  # Show up to 3
            stats_text += f"\n  Bin {bin_idx+1}: p = {bin_p_corrected[bin_idx]:.4f}"
        if len(sig_bins) > 3:
            stats_text += f"\n  ... ({len(sig_bins)-3} more)"
    
    # Add output of KS test
    ks_stat, ks_pval = ks_test_results[cluster_id]
    if not (np.isnan(ks_stat) or np.isnan(ks_pval)):
        stats_text += f"\n\nKS: stat={ks_stat:.3f}, p={ks_pval:.3g}"

    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold',
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Add sample size
    n_dense = len(result['et_dense_neurons'])
    n_sparse = len(result['et_sparse_neurons'])
    n_text = f"n_dense={n_dense}\nn_sparse={n_sparse}"
    ax.text(0.02, 0.98, n_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))
    
    # Legend note
    legend_note = "* = sig (across-all FDR)\n+ = sig (within only)"
    if np.any(bin_rejected) or np.any(bin_rejected_within):
        ax.text(0.5, -0.15, legend_note, 
               transform=ax.transAxes, ha='center', fontsize=8, style='italic')
    
    ax.set_xlabel('Depth Bin (μm)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Proportion of Neurons', fontsize=11, fontweight='bold')
    ax.set_title(f'Cluster {cluster_id}', fontsize=13, fontweight='bold')
    ax.set_xticks(bin_centers)
    ax.set_xticklabels([f'{bin_edges[i]:.0f}-{bin_edges[i+1]:.0f}' for i in range(len(bin_centers))],
                       rotation=45, ha='right')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    max_prop = max(np.max(et_dense_props), np.max(et_sparse_props))
    ax.set_ylim(0, max_prop * 1.25 if max_prop > 0 else 1)

plt.tight_layout()
plt.savefig("depth_et_dense_vs_sparse_by_cluster_fdr.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Detailed statistics printout
print("\n" + "=" * 80)
print("DETAILED STATISTICS BY CLUSTER")
print("=" * 80)

for cluster_id in range(3):
    result = results_by_cluster[cluster_id]
    print(f"\nCluster {cluster_id}:")
    print(f"  ET-dense (vPrCG): {len(result['et_dense_neurons'])} neurons")
    print(f"  ET-sparse (others): {len(result['et_sparse_neurons'])} neurons")
    
    # Print KS statistic for this cluster
    ks_stat, ks_pval = ks_test_results[cluster_id]
    if np.isnan(ks_stat) or np.isnan(ks_pval):
        print("  KS test: Not enough data.")
    else:
        print(f"  KS test: D = {ks_stat:.4f}, p = {ks_pval:.4g}")
    
    print(f"\n  Depth bin statistics (FDR-corrected):")
    print(f"    Bin     ET-dense    ET-sparse    χ²      p(raw)    p(within)  p(across)  Sig")
    print(f"    {'-' * 75}")
    
    for i in range(len(bin_centers)):
        dense_prop = result['et_dense_props'][i]
        sparse_prop = result['et_sparse_props'][i]
        dense_count = int(result['et_dense_counts'][i])
        sparse_count = int(result['et_sparse_counts'][i])
        
        chi2_val = result['bin_chi2_stats'][i]
        p_raw = result['bin_p_values'][i]
        p_within = result['bin_p_corrected_within'][i]
        p_across = result['bin_p_corrected_across'][i]
        sig_within = result['bin_rejected_within'][i]
        sig_across = result['bin_rejected_across'][i]
        
        sig_marker = ""
        if sig_across:
            sig_marker = "*** (across)"
        elif sig_within:
            sig_marker = "** (within)"
        else:
            sig_marker = "ns"
        
        chi2_str = f"{chi2_val:.2f}" if not np.isnan(chi2_val) else "N/A"
        p_raw_str = f"{p_raw:.4f}" if not np.isnan(p_raw) else "N/A"
        p_within_str = f"{p_within:.4f}" if not np.isnan(p_within) else "N/A"
        p_across_str = f"{p_across:.4f}" if not np.isnan(p_across) else "N/A"
        
        print(f"    {i+1:2d}     {dense_prop:.3f} ({dense_count:3d})   {sparse_prop:.3f} ({sparse_count:3d})   {chi2_str:>6s}   {p_raw_str:>8s}   {p_within_str:>8s}   {p_across_str:>8s}   {sig_marker}")
    
    n_sig_within = np.sum(result['bin_rejected_within'])
    n_sig_across = np.sum(result['bin_rejected_across'])
    print(f"\n  Summary:")
    print(f"    Significant bins (within-cluster FDR): {n_sig_within}/{len(bin_centers)}")
    print(f"    Significant bins (across-all-clusters FDR): {n_sig_across}/{len(bin_centers)}")

print("\n" + "=" * 80)
print("FDR CORRECTION EXPLANATION:")
print("=" * 80)
print("""
Within-cluster FDR: Corrects for 6 comparisons (one per bin) within each cluster.
  → Appropriate if each cluster is a separate scientific question
  → Less conservative

Across-all-clusters FDR: Corrects for 18 comparisons (3 clusters × 6 bins) together.
  → Appropriate if comparing all clusters in one analysis
  → More conservative, accounts for multiple cluster testing
  → RECOMMENDED for overall analysis

The visualization shows:
  * = Significant after across-all-clusters FDR correction (more conservative)
  + = Significant only after within-cluster FDR correction
""")
print(f"Depth range: {np.min(depths):.1f} to {np.max(depths):.1f} μm")
print(f"Bin edges (μm): {bin_edges}")
print("=" * 80)

In [ ]:
# ENTER NEW CODE HERE
# ADAPTED FROM R_EI_model.ipynb FOR ASSEMBLY-ONLY DEPTH ANALYSIS
# (keeps core assembly logic; only filtering/KS step is made behavior-duration-aware)

import fnmatch
import numpy as np
import re
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest


def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')


def calculate_mutual_information(x, y, bins=20):
    """Calculate mutual information between two variables using histogram-based approach."""
    valid_mask = ~(np.isnan(x) | np.isnan(y))
    if np.sum(valid_mask) < 2:
        return np.nan

    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    hist_2d, _, _ = np.histogram2d(x_valid, y_valid, bins=bins)
    hist_x = np.sum(hist_2d, axis=1)
    hist_y = np.sum(hist_2d, axis=0)

    p_xy = hist_2d / np.sum(hist_2d)
    p_x = hist_x / np.sum(hist_x)
    p_y = hist_y / np.sum(hist_y)

    mi = 0.0
    for i in range(len(p_x)):
        for j in range(len(p_y)):
            if p_xy[i, j] > 0 and p_x[i] > 0 and p_y[j] > 0:
                mi += p_xy[i, j] * np.log2(p_xy[i, j] / (p_x[i] * p_y[j]))
    return mi


def calculate_information_capacity(data, time_bins, min_samples=10):
    """Calculate information storage capacity using multiple metrics."""
    valid_mask = ~np.isnan(data)
    if np.sum(valid_mask) < min_samples:
        return {
            'entropy': np.nan,
            'mutual_info_temporal': np.nan,
            'capacity': np.nan,
            'n_valid_samples': np.sum(valid_mask),
            'data_range': np.nan,
            'data_std': np.nan,
        }

    valid_data = data[valid_mask]

    n_bins = min(20, len(valid_data) // 5)
    if n_bins < 5:
        n_bins = 5

    hist, _ = np.histogram(valid_data, bins=n_bins)
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]

    if len(prob) < 2:
        entropy_val = np.nan
    else:
        entropy_val = -np.sum(prob * np.log2(prob))

    if len(valid_data) > 1:
        try:
            mi_temporal = calculate_mutual_information(
                valid_data[:-1],
                valid_data[1:],
                bins=min(10, len(valid_data) // 10),
            )
        except Exception:
            mi_temporal = np.nan
    else:
        mi_temporal = np.nan

    if not np.isnan(entropy_val) and not np.isnan(mi_temporal):
        capacity = entropy_val - mi_temporal
    elif not np.isnan(entropy_val):
        capacity = entropy_val
    else:
        capacity = np.nan

    data_range = np.max(valid_data) - np.min(valid_data) if len(valid_data) > 0 else np.nan
    data_std = np.std(valid_data) if len(valid_data) > 0 else np.nan

    return {
        'entropy': entropy_val,
        'mutual_info_temporal': mi_temporal,
        'capacity': capacity,
        'n_valid_samples': np.sum(valid_mask),
        'data_range': data_range,
        'data_std': data_std,
    }


def calculate_ei_balance(assembly_pattern, cluster_labels):
    """
    Calculate E/I balance for an assembly pattern.
    Cluster convention in this notebook: 0=exc, 1=inh, 2=other.
    """
    ei_balance = 0.0
    for neuron_idx, weight in enumerate(assembly_pattern):
        cluster_id = cluster_labels[neuron_idx]
        if cluster_id == 0:
            ei_balance += weight * 1
        elif cluster_id == 1:
            ei_balance += weight * (-1)
        elif cluster_id == 2:
            ei_balance += weight * 0
    return ei_balance


# --- behavior-duration-aware filtering helpers (same pattern as main loop) ---
MIN_BEHAVIOR_MINUTES = 0


def _behavior_duration_seconds(task_times_obj):
    starts = np.asarray(task_times_obj.start, dtype=float).ravel()
    ends = np.asarray(task_times_obj.end, dtype=float).ravel()
    if starts.size != ends.size:
        return np.nan
    return float(np.sum(ends - starts))


def _ks_time_bounds_full_recording(spike_times_obj):
    """Min/max time for uniform KS over full recording."""
    tmin, tmax = np.inf, -np.inf
    for u in range(len(spike_times_obj)):
        idx = spike_times_obj[u].as_series().index.values
        if len(idx):
            tmin = min(tmin, float(np.min(idx)))
            tmax = max(tmax, float(np.max(idx)))
    if np.isfinite(tmin) and np.isfinite(tmax):
        return tmin, tmax
    return np.nan, np.nan


# Get ET-Sparse opercular=0 insertions
et_sparse_op0_mask = np.array([
    neuron['opercular'] == 0
    for neuron in et_sparse_metadata
])

et_sparse_op0_neurons = et_sparse_labels[et_sparse_op0_mask]
et_sparse_op0_metadata = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 0]

# Unique insertion indices for ET-Sparse opercular=0
et_sparse_op0_insertions = sorted({neuron['insertion_idx'] for neuron in et_sparse_op0_metadata})
print(f"ET-Sparse opercular=0 insertions: {et_sparse_op0_insertions}")
print(f"Total insertions to process: {len(et_sparse_op0_insertions)}")


# Create mapping from (nwb_idx, session_idx) to insertion_idx using same key-walk logic
nwb_to_insertion_map = {}
insertion_counter = 0

for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = list(data.keys())

    keys = [key for key in keys if fnmatch.fnmatch(key, "*imec*")]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys

    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [
        key for key in keys
        if not fnmatch.fnmatch(key, template_sentgen)
        and not fnmatch.fnmatch(key, template_auto)
    ]

    if 'NP137' in str(nwb_paths[i]) or 'NP139_B2' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]

    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        nwb_to_insertion_map[(i, s)] = insertion_counter
        insertion_counter += 1

print(f"NWB to insertion mapping created. Total insertions mapped: {insertion_counter}")


# Storage for results used by the next depth-analysis cell
assembly_results = []
neuron_results = []
insertion_results = []


# Process each insertion
for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = list(data.keys())

    keys = [key for key in keys if fnmatch.fnmatch(key, "*imec*")]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys

    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [
        key for key in keys
        if not fnmatch.fnmatch(key, template_sentgen)
        and not fnmatch.fnmatch(key, template_auto)
    ]

    if 'NP137' in str(nwb_paths[i]) or 'NP139_B2' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]

    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        insertion_idx = nwb_to_insertion_map[(i, s)]

        # Skip if this insertion is not in ET-Sparse opercular=0
        if insertion_idx not in et_sparse_op0_insertions:
            continue

        # Guard against mismatch if insertion lists are shorter (e.g., excluded PoCG entries)
        if insertion_idx >= len(region_list):
            print(f"Skipping insertion {insertion_idx}: outside region_list length {len(region_list)}")
            continue

        print(f"\nProcessing insertion {insertion_idx}: {keys[s]}")
        print(f"Region = {region_list[insertion_idx]}")
        print(f"Subject = {subj_list[insertion_idx]}")

        spike_times = data[keys[s]]

        # 1) neuron filtering with flexible full-recording vs behavior-restricted KS window
        firingRates_all = spike_times.metadata["rate"]

        if "TaskTimes" in data.keys():
            task_times = data["TaskTimes"]
        else:
            task_times = data["task_times"]

        start_time = task_times.start
        end_time = task_times.end
        beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)

        beh_sec = _behavior_duration_seconds(task_times)
        beh_min = beh_sec / 60.0 if np.isfinite(beh_sec) else np.nan
        use_full_recording = (not np.isfinite(beh_min)) or (beh_min < MIN_BEHAVIOR_MINUTES)

        if use_full_recording:
            spike_times_beh = spike_times
            firingRates_beh = firingRates_all
            min_time, max_time = _ks_time_bounds_full_recording(spike_times)
            if not (np.isfinite(min_time) and np.isfinite(max_time)) or (max_time <= min_time):
                print(f"  Short behavior ({beh_min:.2f} min) but could not get recording bounds; skipping insertion")
                continue
            print(
                f"  Behavioral duration {beh_min:.2f} min (< {MIN_BEHAVIOR_MINUTES} min) — using full recording for filters; KS window [{min_time:.3f}, {max_time:.3f}] s"
            )
        else:
            spike_times_beh = spike_times.restrict(beh_epochs)
            firingRates_beh = spike_times_beh.metadata["rate"]
            min_time = float(np.asarray(start_time).ravel()[0])
            max_time = float(np.asarray(end_time).ravel()[-1])
            print(
                f"  Behavioral duration {beh_min:.2f} min (>= {MIN_BEHAVIOR_MINUTES} min) — using TaskTimes restrict; KS window [{min_time:.3f}, {max_time:.3f}] s"
            )

        ks_stats = np.zeros(len(spike_times))
        ks_pvals = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            test = spike_times[u].as_series().index.values
            if len(test) > 1:
                normalized_spike_times = (test - min_time) / (max_time - min_time)
                ks_result = kstest(normalized_spike_times, 'uniform')
                ks_stats[u] = ks_result.statistic
                ks_pvals[u] = ks_result.pvalue
            else:
                ks_stats[u] = np.nan
                ks_pvals[u] = np.nan

        violationThreshold = 3 / 1000
        violationPct = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            unit = spike_times[u].as_series().index
            if len(unit) < 100:
                violationPct[u] = 1
            else:
                isi = unit.diff()[1:]
                violations = np.where(isi < violationThreshold)
                violations = np.array(violations)
                violationPct[u] = violations.size / len(isi)

        if "KSLabel" in spike_times.metadata:
            KSLabels = spike_times.metadata["KSLabel"]
        else:
            KSLabels = spike_times.metadata["quality"]

        firingRates = firingRates_beh
        mask1 = violationPct < 3 / 100
        mask2 = firingRates > 0.5
        mask3 = KSLabels != "noise"
        mask4 = ks_stats < 0.3
        mask = mask1 & mask2 & mask3 & mask4

        indicesFinal = firingRates.index[mask]
        indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion_idx])
        indicesFinal = np.sort(indicesFinal)

        spike_times_good = spike_times[indicesFinal]
        print(f"Number of good neurons: {len(spike_times_good)}")

        if len(spike_times_good) == 0:
            print("No good neurons found, skipping insertion")
            continue

        # Match each good unit_id to ET-sparse cluster/metadata deterministically
        insertion_cluster_labels = []
        insertion_metadata = []

        unit_id_to_cluster = {}
        for global_idx, global_neuron in enumerate(et_sparse_metadata):
            if (
                global_neuron['insertion_idx'] == insertion_idx
                and global_neuron['opercular'] == 0
            ):
                unit_id = global_neuron.get('unit_id')
                if unit_id is not None:
                    unit_id_to_cluster[unit_id] = {
                        'global_idx': global_idx,
                        'cluster_label': et_sparse_labels[global_idx],
                        'metadata': global_neuron,
                    }

        for unit_id in indicesFinal:
            if unit_id in unit_id_to_cluster:
                match_info = unit_id_to_cluster[unit_id]
                insertion_cluster_labels.append(match_info['cluster_label'])
                insertion_metadata.append(match_info['metadata'])
            else:
                insertion_cluster_labels.append(-1)
                insertion_metadata.append({})

        insertion_cluster_labels = np.array(insertion_cluster_labels)

        # Build z-scored firing-rate matrix
        timescale = 25 / 1000
        spikeCountMatrix = spike_times_good.count(bin_size=timescale)
        bin_edges = spikeCountMatrix.index.values
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        if len(bin_centers) < len(bin_edges):
            last_width = bin_edges[-1] - bin_edges[-2] if len(bin_edges) > 1 else 0
            last_center = bin_edges[-1] - last_width / 2
            bin_centers = np.append(bin_centers, last_center)

        spikeCountMatrix = spikeCountMatrix.values
        firingRateMatrix = spikeCountMatrix / timescale
        firingRateMatrix = stats.zscore(firingRateMatrix, axis=0)

        # PCA + assembly selection
        pca = PCA()
        firingRateMatrix_pca = pca.fit_transform(firingRateMatrix)
        eigenvalues = pca.explained_variance_
        upperbound = (1 + np.sqrt(firingRateMatrix.shape[1] / firingRateMatrix.shape[0])) ** 2
        assemblyIndices = np.where(eigenvalues > upperbound)
        assemblyEigenvalues = eigenvalues[assemblyIndices]
        print(f"Number of assembly patterns: {len(assemblyEigenvalues)}")

        n_pcs = len(assemblyEigenvalues)
        pc_vectors = pca.components_[:n_pcs, :]
        projections = firingRateMatrix @ pc_vectors.T

        if n_pcs <= 0:
            print(f"No assemblies found for insertion {insertion_idx}")
            continue

        fastica = FastICA(
            n_components=n_pcs,
            algorithm='parallel',
            whiten='unit-variance',
            max_iter=500,
            tol=1e-7,
            random_state=1,
        )
        ica_components = fastica.fit_transform(projections)
        unmixing_matrix = fastica.components_
        ica_assembly_patterns = unmixing_matrix @ pc_vectors

        ap_norm_matrix = ica_assembly_patterns.copy()
        ap_norm_member = np.zeros(ica_assembly_patterns.shape)

        for ii in range(n_pcs):
            ap = ica_assembly_patterns[ii, :]
            ap_norm = ap / norm(ap)
            maxW = np.max(ap_norm)
            minW = np.min(ap_norm)
            if abs(minW) > maxW:
                ap_norm = ap_norm * -1
            ap_norm_matrix[ii, :] = ap_norm

            threshold = np.mean(ap_norm) + np.std(ap_norm) * 2
            for c in range(len(ap)):
                if ap_norm[c] > threshold:
                    ap_norm_member[ii, c] = 1

        sparsity_list = []
        for ap_idx in range(len(ap_norm_matrix)):
            w = np.array(ap_norm_matrix[ap_idx, :])
            n_w = w.size
            numerator = np.sqrt(n_w) - np.sum(np.abs(w))
            denominator = np.sqrt(n_w) - 1
            sparsity = 1 - numerator / denominator if denominator != 0 else np.nan
            sparsity_list.append(sparsity)

        insertion_assembly_results = []

        for assembly_idx in range(n_pcs):
            print(f"Analyzing assembly {assembly_idx + 1} of {n_pcs}")

            assembly_pattern = ap_norm_matrix[assembly_idx, :]
            assembly_members = ap_norm_member[assembly_idx, :]

            ei_balance = calculate_ei_balance(assembly_pattern, insertion_cluster_labels)
            assembly_expression = ica_components[:, assembly_idx]
            assembly_info_capacity = calculate_information_capacity(assembly_expression, bin_centers)

            neuron_details = []
            for neuron_idx in range(len(assembly_pattern)):
                if neuron_idx < len(insertion_metadata) and insertion_metadata[neuron_idx]:
                    depth = insertion_metadata[neuron_idx].get('depth', 0.0)
                    metadata = insertion_metadata[neuron_idx]
                else:
                    depth = 0.0
                    metadata = {}

                neuron_details.append({
                    'neuron_idx': neuron_idx,
                    'assembly_weight': assembly_pattern[neuron_idx],
                    'is_member': bool(assembly_members[neuron_idx]),
                    'cluster_label': insertion_cluster_labels[neuron_idx],
                    'depth': depth,
                    'metadata': metadata,
                })

            assembly_result = {
                'insertion_idx': insertion_idx,
                'assembly_idx': assembly_idx,
                'ei_balance': ei_balance,
                'information_capacity': assembly_info_capacity,
                'sparsity': sparsity_list[assembly_idx],
                'n_neurons': len(assembly_pattern),
                'n_members': int(np.sum(assembly_members)),
                'neuron_details': neuron_details,
            }

            insertion_assembly_results.append(assembly_result)
            assembly_results.append(assembly_result)

        insertion_result = {
            'insertion_idx': insertion_idx,
            'subject': subj_list[insertion_idx],
            'region': region_list[insertion_idx],
            'pathology': path_list[insertion_idx],
            'grade': grade_list[insertion_idx],
            'n_assemblies': n_pcs,
            'n_neurons': len(spike_times_good),
            'assemblies': insertion_assembly_results,
        }

        insertion_results.append(insertion_result)
        print(f"Processed {n_pcs} assemblies and {len(spike_times_good)} neurons for insertion {insertion_idx}")

print("\nAnalysis complete!")
print(f"Total insertions processed: {len(insertion_results)}")
print(f"Total assemblies analyzed: {len(assembly_results)}")

In [ ]:
# DEPTH ANALYSIS FOR UNIQUE NEURONS FROM ASSEMBLY ANALYSIS ONLY
# OLI/AST vs GBM COMPARISONS ONLY
plt.rcParams['svg.fonttype'] = 'none' 

def calculate_depth_distributions_unique(assemblies_data):
    """Calculate depth distributions for unique neurons only"""
    
    # Extract unique neurons (avoid counting the same neuron multiple times)
    unique_neurons = {}
    
    for assembly in assemblies_data:
        for neuron_detail in assembly['neuron_details']:
            if neuron_detail['metadata']:  # Only include neurons with metadata
                # Create unique key: insertion_idx + neuron_idx
                neuron_key = f"{assembly['insertion_idx']}_{neuron_detail['neuron_idx']}"
                
                if neuron_key not in unique_neurons:
                    # First time seeing this neuron
                    neuron_data = {
                        'depth': neuron_detail['depth'],
                        'insertion_idx': assembly['insertion_idx'],
                        'is_member': neuron_detail['is_member'],
                        'cluster_label': neuron_detail['cluster_label']
                    }
                    unique_neurons[neuron_key] = neuron_data
                else:
                    # Neuron already exists - update is_member to True if it's a member in this assembly
                    if neuron_detail['is_member']:
                        unique_neurons[neuron_key]['is_member'] = True
    
    # Convert to list
    all_neurons = list(unique_neurons.values())
    
    # Get insertion metadata from the global lists
    insertion_metadata = {}
    for assembly in assemblies_data:
        insertion_idx = assembly['insertion_idx']
        if insertion_idx not in insertion_metadata:
            # Get pathology and grade from global lists
            insertion_metadata[insertion_idx] = {
                'pathology': path_list[insertion_idx],
                'grade': grade_list[insertion_idx]
            }
    
    # Add pathology and grade to neuron data
    for neuron in all_neurons:
        insertion_idx = neuron['insertion_idx']
        neuron['pathology'] = insertion_metadata[insertion_idx]['pathology']
        neuron['grade'] = insertion_metadata[insertion_idx]['grade']
    
    return all_neurons

def create_depth_bins(depths, n_bins=6):
    """Create depth bins from 0 to max depth"""
    # Handle negative depths by setting them to 0
    depths_clean = np.maximum(depths, 0)
    max_depth = np.max(depths_clean)
    bin_edges = np.linspace(0, max_depth, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    return bin_edges, bin_centers

# Calculate depth distributions from your assembly analysis (unique neurons only)
all_neurons = calculate_depth_distributions_unique(assembly_results)

# Create depth bins
depths = [neuron['depth'] for neuron in all_neurons]
bin_edges, bin_centers = create_depth_bins(depths, n_bins=6)

# Define groups - OLI/AST vs GBM only
oli_ast_neurons = [n for n in all_neurons if n['pathology'] in ['oli', 'ast']]
gbm_neurons = [n for n in all_neurons if n['pathology'] == 'gbm']

print("=== DEPTH ANALYSIS SUMMARY (UNIQUE NEURONS ONLY) ===")
print(f"Total unique neurons: {len(all_neurons)}")
print(f"Oli+Ast neurons: {len(oli_ast_neurons)}")
print(f"GBM neurons: {len(gbm_neurons)}")

# Function to calculate depth bin counts
def get_depth_bin_counts(neurons, bin_edges):
    depths = [n['depth'] for n in neurons]
    # Set negative depths to 0
    depths_clean = np.maximum(depths, 0)
    counts, _ = np.histogram(depths_clean, bins=bin_edges)
    return counts.astype(float)  # Convert to float to avoid casting errors

# Function to get assembly members
def get_assembly_members(neurons):
    return [n for n in neurons if n['is_member']]

# Calculate distributions
oli_ast_all_counts = get_depth_bin_counts(oli_ast_neurons, bin_edges)
gbm_all_counts = get_depth_bin_counts(gbm_neurons, bin_edges)

oli_ast_members = get_assembly_members(oli_ast_neurons)
gbm_members = get_assembly_members(gbm_neurons)

oli_ast_member_counts = get_depth_bin_counts(oli_ast_members, bin_edges)
gbm_member_counts = get_depth_bin_counts(gbm_members, bin_edges)

# Calculate fractions (now with float arrays)
oli_ast_fractions = np.divide(oli_ast_member_counts, oli_ast_all_counts, 
                             out=np.zeros_like(oli_ast_member_counts, dtype=float), where=oli_ast_all_counts!=0)
gbm_fractions = np.divide(gbm_member_counts, gbm_all_counts, 
                          out=np.zeros_like(gbm_member_counts, dtype=float), where=gbm_all_counts!=0)

# Create visualization with VERTICAL layout: smaller overall size, reduced top plot
fig = plt.figure(figsize=(10, 8))

# Create custom grid layout - VERTICAL: 3 rows, 1 column with height ratios (reduced top plot)
gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1], hspace=0.3, wspace=0.3)

# First plot: All neurons and assembly members on same axes (takes up 2/3 of space)
ax1 = fig.add_subplot(gs[:2, 0])

# Plot all neurons
ax1.plot(bin_centers, oli_ast_all_counts, 'o-', label='Oli+Ast (All Neurons)', color='blue', linewidth=2, markersize=0)
ax1.plot(bin_centers, gbm_all_counts, 'o-', label='GBM (All Neurons)', color='red', linewidth=2, markersize=0)

# Plot assembly members
ax1.plot(bin_centers, oli_ast_member_counts, 'o--', label='Oli+Ast (Members)', color='blue', linewidth=2, markersize=0, alpha=0.7)
ax1.plot(bin_centers, gbm_member_counts, 'o--', label='GBM (Members)', color='red', linewidth=2, markersize=0, alpha=0.7)

ax1.set_xlabel('Depth Bin')
ax1.set_ylabel('Number of Neurons')
ax1.set_title('Depth Distribution: All Neurons vs Assembly Members\nOli+Ast vs GBM')
ax1.set_xticks(bin_centers)
ax1.set_xticklabels([f'{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}' for i in range(len(bin_centers))])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Second plot: Membership fraction (takes up 1/3 of space)
ax2 = fig.add_subplot(gs[2, 0])

ax2.plot(bin_centers, oli_ast_fractions, 'o-', label='Oli+Ast', color='blue', linewidth=2, markersize=0)
ax2.plot(bin_centers, gbm_fractions, 'o-', label='GBM', color='red', linewidth=2, markersize=0)
ax2.set_xlabel('Depth Bin')
ax2.set_ylabel('Fraction of Members/All Neurons')
ax2.set_title('Assembly Membership Fraction\nOli+Ast vs GBM')
ax2.set_xticks(bin_centers)
ax2.set_xticklabels([f'{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}' for i in range(len(bin_centers))])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("depth_distribution.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Statistical analysis (with error handling)
print("=== STATISTICAL ANALYSIS ===")

# Chi-square tests for depth distributions
from scipy.stats import chi2_contingency, fisher_exact, ttest_ind, ks_2samp, mannwhitneyu
import pandas as pd

def chi_square_test(group1_counts, group2_counts, group1_name, group2_name):
    """Perform chi-square test for depth distributions"""
    # Create contingency table
    contingency_table = np.array([group1_counts, group2_counts])
    
    # Check for zero expected frequencies
    row_totals = np.sum(contingency_table, axis=1)
    col_totals = np.sum(contingency_table, axis=0)
    total = np.sum(contingency_table)
    
    expected = np.outer(row_totals, col_totals) / total
    
    if np.any(expected == 0):
        print(f"\n{group1_name} vs {group2_name}:")
        print(f"⚠️ Cannot perform chi-square test - zero expected frequencies")
        print(f"Group 1 counts: {group1_counts}")
        print(f"Group 2 counts: {group2_counts}")
        return
    
    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    print(f"\n{group1_name} vs {group2_name}:")
    print(f"Chi-square statistic: {chi2:.3f}")
    print(f"p-value: {p_value:.3f}")
    print(f"Degrees of freedom: {dof}")
    
    if p_value < 0.05:
        print(f"✓ Significant difference in depth distributions")
    else:
        print(f"✗ No significant difference in depth distributions")

def test_membership_fraction_differences(group1_fractions, group2_fractions, 
                                       group1_member_counts, group1_all_counts,
                                       group2_member_counts, group2_all_counts,
                                       group1_name, group2_name, bin_edges):
    """Test if membership fractions differ between groups using KS test and Mann-Whitney U"""
    print(f"\n=== MEMBERSHIP FRACTION DIFFERENCES: {group1_name} vs {group2_name} ===")
    
    # Show the fractions for each bin
    print("Membership fractions by depth bin:")
    for i in range(len(bin_edges)-1):
        bin_range = f"{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}"
        
        frac1 = group1_fractions[i]
        frac2 = group2_fractions[i]
        
        print(f"Bin {i+1} ({bin_range}):")
        print(f"  {group1_name}: {frac1:.3f}")
        print(f"  {group2_name}: {frac2:.3f}")
        print(f"  Difference: {frac1 - frac2:.3f}")
    
    # Remove any NaN values for testing
    frac1_clean = group1_fractions[~np.isnan(group1_fractions)]
    frac2_clean = group2_fractions[~np.isnan(group2_fractions)]
    
    print(f"\nStatistical tests on membership fractions:")
    
    # Kolmogorov-Smirnov test
    try:
        ks_statistic, ks_p_value = ks_2samp(frac1_clean, frac2_clean)
        print(f"Kolmogorov-Smirnov test:")
        print(f"  KS statistic: {ks_statistic:.3f}")
        print(f"  p-value: {ks_p_value:.3f}")
        
        if ks_p_value < 0.05:
            print(f"  ✓ Significant difference in membership fraction distributions")
        else:
            print(f"  ✗ No significant difference in membership fraction distributions")
    except Exception as e:
        print(f"  Could not perform KS test - {e}")
    
    # Mann-Whitney U test
    try:
        mw_statistic, mw_p_value = mannwhitneyu(frac1_clean, frac2_clean, alternative='two-sided')
        print(f"Mann-Whitney U test:")
        print(f"  U statistic: {mw_statistic:.3f}")
        print(f"  p-value: {mw_p_value:.3f}")
        
        if mw_p_value < 0.05:
            print(f"  ✓ Significant difference in membership fraction distributions")
        else:
            print(f"  ✗ No significant difference in membership fraction distributions")
    except Exception as e:
        print(f"  Could not perform Mann-Whitney U test - {e}")
    
    # Also test each bin individually
    print(f"\nIndividual bin tests:")
    for i in range(len(bin_edges)-1):
        bin_range = f"{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}"
        
        # Get counts for this bin
        group1_members = group1_member_counts[i]
        group1_non_members = group1_all_counts[i] - group1_member_counts[i]
        group2_members = group2_member_counts[i]
        group2_non_members = group2_all_counts[i] - group2_member_counts[i]
        
        # Create 2x2 contingency table for this bin
        bin_contingency = np.array([[group1_members, group1_non_members],
                                   [group2_members, group2_non_members]])
        
        try:
            odds_ratio, p_value = fisher_exact(bin_contingency)
            
            print(f"Bin {i+1} ({bin_range}):")
            print(f"  {group1_name}: {group1_members}/{group1_all_counts[i]} = {group1_members/group1_all_counts[i]:.3f}")
            print(f"  {group2_name}: {group2_members}/{group2_all_counts[i]} = {group2_members/group2_all_counts[i]:.3f}")
            print(f"  Fisher's exact test: OR = {odds_ratio:.3f}, p = {p_value:.3f}")
            
            if p_value < 0.05:
                print(f"  ✓ Significant difference in this bin")
            else:
                print(f"  ✗ No significant difference in this bin")
        except Exception as e:
            print(f"Bin {i+1} ({bin_range}): Could not perform test - {e}")

def test_individual_depth_bins(group1_counts, group2_counts, group1_name, group2_name, bin_edges):
    """Test each depth bin individually using Fisher's exact test"""
    print(f"\n=== INDIVIDUAL DEPTH BIN TESTS: {group1_name} vs {group2_name} ===")
    
    for i in range(len(group1_counts)):
        bin_range = f"{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}"
        
        # Create 2x2 contingency table for this bin
        # Group 1: count in this bin, count in all other bins
        # Group 2: count in this bin, count in all other bins
        group1_this_bin = group1_counts[i]
        group1_other_bins = np.sum(group1_counts) - group1_this_bin
        group2_this_bin = group2_counts[i]
        group2_other_bins = np.sum(group2_counts) - group2_this_bin
        
        # Create contingency table
        contingency_table = np.array([[group1_this_bin, group1_other_bins],
                                     [group2_this_bin, group2_other_bins]])
        
        # Perform Fisher's exact test
        try:
            odds_ratio, p_value = fisher_exact(contingency_table)
            
            print(f"Bin {i+1} ({bin_range}):")
            print(f"  {group1_name}: {group1_this_bin}/{np.sum(group1_counts)} = {group1_this_bin/np.sum(group1_counts):.3f}")
            print(f"  {group2_name}: {group2_this_bin}/{np.sum(group2_counts)} = {group2_this_bin/np.sum(group2_counts):.3f}")
            print(f"  Odds ratio: {odds_ratio:.3f}")
            print(f"  p-value: {p_value:.3f}")
            
            if p_value < 0.05:
                print(f"  ✓ Significant difference in this bin")
            else:
                print(f"  ✗ No significant difference in this bin")
                
        except Exception as e:
            print(f"Bin {i+1} ({bin_range}): Could not perform test - {e}")

# Test all neurons distributions
chi_square_test(oli_ast_all_counts, gbm_all_counts, "Oli+Ast", "GBM")

# Test assembly members distributions
chi_square_test(oli_ast_member_counts, gbm_member_counts, "Oli+Ast Members", "GBM Members")

# MOST IMPORTANT: Test if membership fractions differ between groups
test_membership_fraction_differences(oli_ast_fractions, gbm_fractions, 
                                   oli_ast_member_counts, oli_ast_all_counts,
                                   gbm_member_counts, gbm_all_counts,
                                   "Oli+Ast", "GBM", bin_edges)

# Test individual depth bins
test_individual_depth_bins(oli_ast_all_counts, gbm_all_counts, "Oli+Ast", "GBM", bin_edges)
test_individual_depth_bins(oli_ast_member_counts, gbm_member_counts, "Oli+Ast Members", "GBM Members", bin_edges)

# Summary statistics
print(f"\n=== SUMMARY STATISTICS ===")
print(f"Oli+Ast - Total: {len(oli_ast_neurons)}, Members: {len(oli_ast_members)}, Fraction: {len(oli_ast_members)/len(oli_ast_neurons):.3f}")
print(f"GBM - Total: {len(gbm_neurons)}, Members: {len(gbm_members)}, Fraction: {len(gbm_members)/len(gbm_neurons):.3f}")

# Depth range analysis
print(f"\n=== DEPTH RANGE ANALYSIS ===")
print(f"All neurons depth range: {np.min(depths):.1f} to {np.max(depths):.1f}")
print(f"Bin edges: {bin_edges}")
print(f"Bin centers: {bin_centers}")